# Walmart — IF Anomaly-Detection Approach (Standalone Replication)

**Adapted from `anomolydetc (1).ipynb`.** Implements her 4/8/12-week moving-average baselines, store-type MAD (median − 2 × MAD), $1,000 department deficit screen, and Isolation Forest (`100` trees, `contamination=0.05`, `random_state=42`).

**Evaluation improvement:** historical-only threshold fitting and Isolation Forest training before **2012-07-20**, evaluation on **2012-07-20 to 2012-10-26**. This is intentionally **not an exact reproduction** of the original full-period-fitting notebook. Sales baselines use only preceding weeks. Markdown missingness is preserved, and the filled-zero total is clearly labeled as a modeling convention, not proof of no markdown activity.

Run top to bottom. Requires the Walmart `train.csv`, `features.csv`, `stores.csv` in your Drive folder. Saves CSVs for merging with your supervised output.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from IPython.display import display

folder = '/content/drive/MyDrive/Walmart Recruiting Sales Data'
output_dir = os.path.join(folder, 'michaela_replication')
os.makedirs(output_dir, exist_ok=True)

train = pd.read_csv(os.path.join(folder, 'train.csv'), parse_dates=['Date'])
features = pd.read_csv(os.path.join(folder, 'features.csv'), parse_dates=['Date'])
stores = pd.read_csv(os.path.join(folder, 'stores.csv'))
assert not features.duplicated(['Store', 'Date']).any()
assert not stores.duplicated(['Store']).any()
assert not train.duplicated(['Store', 'Dept', 'Date']).any()

cutoff = pd.Timestamp('2012-07-20')
test_end = pd.Timestamp('2012-10-26')
markdown_cols = [f'MarkDown{i}' for i in range(1, 6)]

# Store-week feature rows, missing markdowns remain NaN
sw = (train.groupby(['Store', 'Date'], as_index=False)
      .agg(Weekly_Sales=('Weekly_Sales', 'sum'))
      .merge(features, on=['Store', 'Date'], how='left', validate='one_to_one')
      .merge(stores[['Store','Type','Size']], on='Store', how='left', validate='many_to_one')
      .sort_values(['Store','Date']).reset_index(drop=True))

sw['Markdown_Recorded'] = sw[markdown_cols].notna().any(axis=1)
sw['Recorded_Markdown_Count'] = sw[markdown_cols].notna().sum(axis=1)
# Michaela's modeling convention: missing markdown amount -> 0 when summing
sw['Total_MarkDown_Model'] = sw[markdown_cols].fillna(0).sum(axis=1)
sw['Has_Positive_Markdown'] = (sw[markdown_cols] > 0).any(axis=1)

print('Store-weeks:', len(sw), 'Stores:', sw.Store.nunique())
display(sw.head(3))

Mounted at /content/drive
Store-weeks: 6435 Stores: 45


,Store,Date,Weekly_Sales,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday,Type,Size,Markdown_Recorded,Recorded_Markdown_Count,Total_MarkDown_Model,Has_Positive_Markdown
0,1,2010-02-05,1643690.90,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315,False,0,0.0,False
1,1,2010-02-12,1641957.44,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315,False,0,0.0,False
2,1,2010-02-19,1611968.17,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False,A,151315,False,0,0.0,False


In [3]:
# Store-week moving averages: shift first (exclude current week's actual sales)
for window in [4,8,12]:
    sw[f'Store_MA{window}'] = sw.groupby('Store')['Weekly_Sales'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=window).mean()
    )
    sw[f'Store_MA{window}_Shortfall'] = (
        sw[f'Store_MA{window}'] - sw['Weekly_Sales']
    ).clip(lower=0)

sw['Store_YoY_52W'] = sw.groupby('Store')['Weekly_Sales'].shift(52)

# Department-week moving averages and Michaela's type-MAD screening
# Join store-week context onto department sales without changing sales granularity
cols = ['Store','Date','Type','Markdown_Recorded','Has_Positive_Markdown']
dw = (train[['Store','Dept','Date','Weekly_Sales']]
      .merge(sw[cols], on=['Store','Date'], how='left', validate='many_to_one')
      .sort_values(['Store','Dept','Date']).reset_index(drop=True))
for window in [4,8,12]:
    dw[f'Dept_MA{window}'] = dw.groupby(['Store','Dept'])['Weekly_Sales'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    dw[f'Dept_MA{window}_Shortfall'] = (
        dw[f'Dept_MA{window}'] - dw['Weekly_Sales']
    ).clip(lower=0)

# Michaela's 4-week baseline includes up to four historical observations;
# retain only >= $500 expected sales for percentage-based screening.
dw['Valid_MA4'] = dw['Dept_MA4'].ge(500)
dw['Dev_MA4'] = np.where(dw['Valid_MA4'],
                         (dw['Weekly_Sales']-dw['Dept_MA4'])/dw['Dept_MA4'],
                         np.nan)

# Chronological store-type MAD calibration on all historically eligible dept-week rows
cal = dw.loc[(dw['Date'] < cutoff) & dw['Valid_MA4']].copy()
thresholds = {}
for typ in ['A','B','C']:
    values = cal.loc[cal['Type'].eq(typ),'Dev_MA4'].dropna()
    if values.empty: raise ValueError(f'No historical calibration rows for type {typ}')
    med = values.median()
    mad = (values-med).abs().median()
    thresholds[typ] = max(-1., med-2*mad)
print('Historical Type MAD thresholds:', {k: round(v*100,2) for k,v in thresholds.items()})

# Flag only weeks with *recorded positive markdown* and >= $1000 modeled gap;
# preserve the source limitation: markdown data is store-week, not department-specific.
dw['MA4_TypeMAD_Flag'] = (
    dw['Valid_MA4']
    & dw['Has_Positive_Markdown'].fillna(False)
    & dw['Dev_MA4'].lt(dw['Type'].map(thresholds))
    & dw['Dept_MA4_Shortfall'].ge(1000)
)

dw_test = dw.loc[dw['Date'].between(cutoff,test_end)].copy()
dw_test.to_csv(os.path.join(output_dir,'ma_department_week.csv'), index=False)
print('Department test rows:',len(dw_test), 'Type-MAD flags:', int(dw_test.MA4_TypeMAD_Flag.sum()))

Historical Type MAD thresholds: {'A': -20.36, 'B': -22.53, 'C': -14.28}
Department test rows: 44415 Type-MAD flags: 2331


In [4]:
# Isolation Forest, all 45 stores, Michaela's same six feature categories
if_features = ['Weekly_Sales','Total_MarkDown_Model','Temperature','Fuel_Price','CPI','Unemployment']
if_train = sw.loc[(sw.Date < cutoff)].dropna(subset=if_features).copy()
if_test = sw.loc[sw.Date.between(cutoff,test_end)].dropna(subset=if_features).copy()

iso = IsolationForest(n_estimators=100, contamination=0.05,
                      random_state=42, n_jobs=-1)
iso.fit(if_train[if_features])
if_test['IF_Flag'] = iso.predict(if_test[if_features]) == -1
if_test['IF_Anomaly_Score'] = iso.decision_function(if_test[if_features])
print('IF train weeks:',len(if_train),'IF test weeks:',len(if_test),
      'IF flagged:',int(if_test.IF_Flag.sum()),
      'flag rate:',round(100*if_test.IF_Flag.mean(),2),'%')

IF train weeks: 5760 IF test weeks: 675 IF flagged: 68 flag rate: 10.07 %


In [5]:
# Merge department-level type-MAD summary into store-week IF results
summary = (dw_test.groupby(['Store','Date'],as_index=False)
           .agg(MA_TypeMAD_Any=('MA4_TypeMAD_Flag','any'),
                MA_TypeMAD_Dept_Count=('MA4_TypeMAD_Flag','sum'),
                Dept_Count=('Dept','nunique')))
flagged = dw_test.assign(Flagged_MA4_Dollars=np.where(
    dw_test['MA4_TypeMAD_Flag'],dw_test['Dept_MA4_Shortfall'],0))
impact = flagged.groupby(['Store','Date'],as_index=False).agg(
    MA_TypeMAD_Flagged_Dollars=('Flagged_MA4_Dollars','sum'))

export_cols = ['Store','Date','Type','Weekly_Sales','IF_Flag','IF_Anomaly_Score',
               'Markdown_Recorded','Has_Positive_Markdown','Total_MarkDown_Model',
               'Store_MA4','Store_MA8','Store_MA12',
               'Store_MA4_Shortfall','Store_MA8_Shortfall','Store_MA12_Shortfall']
out = (if_test[export_cols]
       .merge(summary,on=['Store','Date'],how='left',validate='one_to_one')
       .merge(impact,on=['Store','Date'],how='left',validate='one_to_one'))
out = out.rename(columns={'Weekly_Sales':'Store_Actual_Sales'})
assert not out.duplicated(['Store','Date']).any()
assert out[['MA_TypeMAD_Any','MA_TypeMAD_Dept_Count']].notna().all().all()
out['Date'] = out['Date'].dt.strftime('%Y-%m-%d')
output = os.path.join(output_dir,'michaela_store_week_comparison.csv')
out.to_csv(output,index=False)
print('Saved:', output, '| rows:',len(out))
display(out.head())

# Summary for store-type analysis
summary_by_type = out.groupby('Type').agg(
    Store_Weeks=('Store','size'),
    IF_Flags=('IF_Flag','sum'),
    MA_TypeMAD_Flags=('MA_TypeMAD_Any','sum'),
    MA4_Estimated_Shortfall=('Store_MA4_Shortfall','sum'),
    MA8_Estimated_Shortfall=('Store_MA8_Shortfall','sum'))
summary_by_type['IF_Flag_Rate_Pct'] = 100*summary_by_type['IF_Flags']/summary_by_type['Store_Weeks']
summary_by_type['MA_TypeMAD_Flag_Rate_Pct'] = 100*summary_by_type['MA_TypeMAD_Flags']/summary_by_type['Store_Weeks']
summary_by_type.to_csv(os.path.join(output_dir,'michaela_by_store_type.csv'))
display(summary_by_type.round(2))

Saved: /content/drive/MyDrive/Walmart Recruiting Sales Data/michaela_replication/michaela_store_week_comparison.csv | rows: 675


,Store,Date,Type,Store_Actual_Sales,IF_Flag,IF_Anomaly_Score,Markdown_Recorded,Has_Positive_Markdown,Total_MarkDown_Model,Store_MA4,Store_MA8,Store_MA12,Store_MA4_Shortfall,Store_MA8_Shortfall,Store_MA12_Shortfall,MA_TypeMAD_Any,MA_TypeMAD_Dept_Count,Dept_Count,MA_TypeMAD_Flagged_Dollars
0,1,2012-07-20,A,1497954.76,False,0.060365,True,True,9026.46,1.591284e+06,1.609112e+06,1.602778e+06,93329.1150,111157.18875,104823.729167,True,7,72,41165.3800
1,1,2012-07-27,A,1439123.71,False,0.033939,True,True,22130.24,1.583811e+06,1.601926e+06,1.605197e+06,144687.4025,162802.01500,166073.645000,True,6,72,38402.0450
2,1,2012-08-03,A,1631135.79,True,-0.016860,True,True,44403.74,1.558487e+06,1.578756e+06,1.584748e+06,0.0000,0.00000,0.000000,False,0,71,0.0000
3,1,2012-08-10,A,1592409.97,False,0.035961,True,True,23488.55,1.523807e+06,1.570495e+06,1.586418e+06,0.0000,0.00000,0.000000,True,2,72,5997.3325
4,1,2012-08-17,A,1597868.05,False,0.060398,True,True,10003.11,1.540156e+06,1.565720e+06,1.586127e+06,0.0000,0.00000,0.000000,True,1,72,7562.2225


,Store_Weeks,IF_Flags,MA_TypeMAD_Flags,MA4_Estimated_Shortfall,MA8_Estimated_Shortfall,IF_Flag_Rate_Pct,MA_TypeMAD_Flag_Rate_Pct
Type,,,,,,,
A,330,48,292,12018355.27,13756196.10,14.55,88.48
B,255,20,230,6635299.12,7828870.19,7.84,90.20
C,90,0,32,846191.83,841238.75,0.00,35.56


In [6]:

# ============================================================
# FINAL — MICHAELA ORIGINAL VS CHRONOLOGICAL MAD
# + ISOLATION FOREST + EXTRA TREES COMPARISON
# ============================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

folder = "/content/drive/MyDrive/Walmart Recruiting Sales Data"
output_dir = os.path.join(folder, "michaela_replication")
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. MICHAELA'S ORIGINAL STORE-TYPE MAD THRESHOLDS
# Decimal form: -20.04% = -0.2004
# ------------------------------------------------------------
original_thresholds = {
    "A": -0.2004,
    "B": -0.2216,
    "C": -0.1404
}

# Read department-level results saved by the replication notebook
dw_compare = pd.read_csv(
    os.path.join(output_dir, "ma_department_week.csv"),
    parse_dates=["Date"]
)

# Preserve same comparison conditions for both threshold versions
for col in ["Valid_MA4", "Has_Positive_Markdown"]:
    dw_compare[col] = (
        dw_compare[col].astype(str).str.lower()
        .map({"true": True, "false": False})
    )
    assert dw_compare[col].notna().all(), f"Invalid values: {col}"

# Original thresholds, applied to the common final-test sample
dw_compare["MA_TypeMAD_Original_Flag"] = (
    dw_compare["Valid_MA4"]
    & dw_compare["Has_Positive_Markdown"]
    & (
        dw_compare["Dev_MA4"]
        < dw_compare["Type"].map(original_thresholds)
    )
    & (dw_compare["Dept_MA4_Shortfall"] >= 1000)
)

# Chronological threshold flags from the previous notebook
dw_compare["MA_TypeMAD_Chronological_Flag"] = (
    dw_compare["MA4_TypeMAD_Flag"]
    .astype(str).str.lower()
    .map({"true": True, "false": False})
)

assert dw_compare[
    "MA_TypeMAD_Chronological_Flag"
].notna().all()

# ------------------------------------------------------------
# 2. AGGREGATE MAD FLAGS TO STORE-WEEK
# ------------------------------------------------------------
dw_compare["Original_Flagged_Dollars"] = np.where(
    dw_compare["MA_TypeMAD_Original_Flag"],
    dw_compare["Dept_MA4_Shortfall"],
    0
)

dw_compare["Chronological_Flagged_Dollars"] = np.where(
    dw_compare["MA_TypeMAD_Chronological_Flag"],
    dw_compare["Dept_MA4_Shortfall"],
    0
)

ma_summary = (
    dw_compare.groupby(["Store", "Date"], as_index=False)
    .agg(
        MA_TypeMAD_Original_Flag=(
            "MA_TypeMAD_Original_Flag", "any"
        ),
        MA_TypeMAD_Chronological_Flag=(
            "MA_TypeMAD_Chronological_Flag", "any"
        ),
        MA_Original_Dept_Count=(
            "MA_TypeMAD_Original_Flag", "sum"
        ),
        MA_Chronological_Dept_Count=(
            "MA_TypeMAD_Chronological_Flag", "sum"
        ),
        MA_Original_Flagged_Dollars=(
            "Original_Flagged_Dollars", "sum"
        ),
        MA_Chronological_Flagged_Dollars=(
            "Chronological_Flagged_Dollars", "sum"
        )
    )
)

# ------------------------------------------------------------
# 3. LOAD MICHAELA'S ISOLATION FOREST RESULTS
# ------------------------------------------------------------
michaela = pd.read_csv(
    os.path.join(
        output_dir, "michaela_store_week_comparison.csv"
    ),
    parse_dates=["Date"]
)

michaela = michaela.drop(
    columns=[
        "MA_TypeMAD_Any",
        "MA_TypeMAD_Dept_Count",
        "MA_TypeMAD_Flagged_Dollars"
    ],
    errors="ignore"
)

michaela = michaela.merge(
    ma_summary,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 4. LOAD YOUR EXTRA TREES + MAD STORE-WEEK RESULTS
# ------------------------------------------------------------
et = pd.read_csv(
    os.path.join(
        folder, "et_mad_store_week_with_factors.csv"
    ),
    parse_dates=["Date"]
)

# Avoid duplicate store-type and external-factor columns
et_cols = [
    "Store", "Date", "Type",
    "ET_MAD_Flag",
    "ET_Net_Shortfall",
    "ET_Flagged_Shortfall",
    "Eligible_Dept_Count",
    "GlobalMAD_Dept_Count",
    "TypeMAD_Any",
    "Fixed10_Any",
    "P90_Any",
    "P95_Any"
]

et = et[et_cols].copy()

assert not et.duplicated(["Store", "Date"]).any()
assert not michaela.duplicated(["Store", "Date"]).any()

# Only weeks evaluated by BOTH pipelines
combined = michaela.merge(
    et.drop(columns=["Type"]),
    on=["Store", "Date"],
    how="inner",
    validate="one_to_one"
)

assert len(combined) > 0, "No matching store-weeks."

# ------------------------------------------------------------
# 5. MODEL AGREEMENT
# ------------------------------------------------------------
flag_cols = [
    "ET_MAD_Flag",
    "IF_Flag",
    "MA_TypeMAD_Original_Flag",
    "MA_TypeMAD_Chronological_Flag"
]

for col in flag_cols:
    combined[col] = (
        combined[col].astype(str).str.lower()
        .map({"true": True, "false": False})
    )
    assert combined[col].notna().all(), f"Invalid {col}"

combined["Both_Flagged"] = (
    combined["ET_MAD_Flag"] & combined["IF_Flag"]
)

combined["ET_Only"] = (
    combined["ET_MAD_Flag"] & ~combined["IF_Flag"]
)

combined["IF_Only"] = (
    ~combined["ET_MAD_Flag"] & combined["IF_Flag"]
)

combined["Agreement_Group"] = np.select(
    [
        combined["Both_Flagged"],
        combined["ET_Only"],
        combined["IF_Only"]
    ],
    [
        "Both flagged",
        "Extra Trees only",
        "Isolation Forest only"
    ],
    default="Neither"
)

def jaccard(a, b):
    intersection = (a & b).sum()
    union = (a | b).sum()
    return intersection / union if union else np.nan

# ------------------------------------------------------------
# 6. BENCHMARK SUMMARY
# ------------------------------------------------------------
methods = {
    "ET + Global MAD": "ET_MAD_Flag",
    "Isolation Forest": "IF_Flag",
    "MA + Original Type MAD": "MA_TypeMAD_Original_Flag",
    "MA + Chronological Type MAD":
        "MA_TypeMAD_Chronological_Flag"
}

benchmark = pd.DataFrame([
    {
        "Method": name,
        "Store_Weeks": len(combined),
        "Flagged_Weeks": int(combined[col].sum()),
        "Flag_Rate_Pct": 100 * combined[col].mean(),
        "Jaccard_vs_ET_GlobalMAD":
            jaccard(
                combined[col],
                combined["ET_MAD_Flag"]
            )
    }
    for name, col in methods.items()
])

print("\nMODEL BENCHMARK COMPARISON")
display(benchmark.round(3))

print("\nET + MAD VS ISOLATION FOREST")
print(
    "Jaccard:",
    round(
        jaccard(
            combined["ET_MAD_Flag"],
            combined["IF_Flag"]
        ), 3
    )
)

display(combined["Agreement_Group"].value_counts())

# ------------------------------------------------------------
# 7. STORE-TYPE COMPARISON
# ------------------------------------------------------------
store_type_rows = []

for typ, group in combined.groupby("Type"):
    for name, col in methods.items():
        store_type_rows.append({
            "Store_Type": typ,
            "Method": name,
            "Store_Weeks": len(group),
            "Flagged_Weeks": int(group[col].sum()),
            "Flag_Rate_Pct": 100 * group[col].mean()
        })

by_type = pd.DataFrame(store_type_rows)

print("\nFLAGGING RATES BY STORE TYPE")
display(by_type.round(2))

# ------------------------------------------------------------
# 8. SAVE ALL RESULTS
# ------------------------------------------------------------
combined.to_csv(
    os.path.join(output_dir, "combined_anomaly_comparison.csv"),
    index=False
)

benchmark.to_csv(
    os.path.join(output_dir, "combined_model_benchmarks.csv"),
    index=False
)

by_type.to_csv(
    os.path.join(output_dir, "combined_comparison_by_store_type.csv"),
    index=False
)

dw_compare.to_csv(
    os.path.join(output_dir, "ma_department_week_both_thresholds.csv"),
    index=False
)

print("\nCOMPLETED SUCCESSFULLY")
print("Matched store-weeks:", len(combined))
print("Files saved to:", output_dir)



MODEL BENCHMARK COMPARISON


,Method,Store_Weeks,Flagged_Weeks,Flag_Rate_Pct,Jaccard_vs_ET_GlobalMAD
0,ET + Global MAD,675,672,99.556,1.000
1,Isolation Forest,675,68,10.074,0.100
2,MA + Original Type MAD,675,557,82.519,0.821
3,MA + Chronological Type MAD,675,554,82.074,0.816



ET + MAD VS ISOLATION FOREST
Jaccard: 0.1


,count
Agreement_Group,
Extra Trees only,605
Both flagged,67
Neither,2
Isolation Forest only,1



FLAGGING RATES BY STORE TYPE


,Store_Type,Method,Store_Weeks,Flagged_Weeks,Flag_Rate_Pct
0,A,ET + Global MAD,330,328,99.39
1,A,Isolation Forest,330,48,14.55
2,A,MA + Original Type MAD,330,293,88.79
3,A,MA + Chronological Type MAD,330,292,88.48
4,B,ET + Global MAD,255,254,99.61
5,B,Isolation Forest,255,20,7.84
6,B,MA + Original Type MAD,255,231,90.59
7,B,MA + Chronological Type MAD,255,230,90.20
8,C,ET + Global MAD,90,90,100.00
9,C,Isolation Forest,90,0,0.00



COMPLETED SUCCESSFULLY
Matched store-weeks: 675
Files saved to: /content/drive/MyDrive/Walmart Recruiting Sales Data/michaela_replication


In [7]:

# ============================================================
# CORRECT STORE-WEEK EXTRA TREES FLAG AGGREGATION
# NO MODEL RETRAINING
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

# Uses the 'combined' DataFrame from your previous cell
df = combined.copy()

# Existing dept-level breadth information
df["ET_Flagged_Dept_Rate"] = (
    df["GlobalMAD_Dept_Count"] /
    df["Eligible_Dept_Count"]
)

# Require aggregate underperformance as well
df["Store_Underperformance"] = (
    df["ET_Expected_Eligible_Sales"] >
    df["ET_Actual_Eligible_Sales"]
) if {
    "ET_Expected_Eligible_Sales",
    "ET_Actual_Eligible_Sales"
}.issubset(df.columns) else (
    df["ET_Net_Shortfall"] > 0
)

# Alternative definitions
df["ET_Breadth10_Flag"] = (
    (df["ET_Flagged_Dept_Rate"] >= 0.10)
    & df["Store_Underperformance"]
)

df["ET_Breadth20_Flag"] = (
    (df["ET_Flagged_Dept_Rate"] >= 0.20)
    & df["Store_Underperformance"]
)

# Compare with Isolation Forest
def jaccard(a, b):
    union = (a | b).sum()
    return (a & b).sum() / union if union else np.nan

flags = {
    "Original Any-Department": "ET_MAD_Flag",
    "Breadth ≥10%": "ET_Breadth10_Flag",
    "Breadth ≥20%": "ET_Breadth20_Flag"
}

rows = []

for name, col in flags.items():
    rows.append({
        "Method": name,
        "Flagged_Store_Weeks": int(df[col].sum()),
        "Flag_Rate_Pct": 100 * df[col].mean(),
        "Jaccard_vs_IF": jaccard(
            df[col].astype(bool),
            df["IF_Flag"].astype(bool)
        )
    })

results = pd.DataFrame(rows)

display(results.round(3))

# Save updated sensitivity results
results.to_csv(
    "/content/drive/MyDrive/Walmart Recruiting Sales Data/"
    "michaela_replication/store_week_flag_sensitivity.csv",
    index=False
)

print("Store-week sensitivity analysis completed.")


,Method,Flagged_Store_Weeks,Flag_Rate_Pct,Jaccard_vs_IF
0,Original Any-Department,672,99.556,0.100
1,Breadth ≥10%,251,37.185,0.074
2,Breadth ≥20%,41,6.074,0.028


Store-week sensitivity analysis completed.


In [8]:

# ============================================================
# STAGE 12 — TOP-20 BUSINESS-PRIORITY COMPARISON
# Extra Trees + Global MAD vs Isolation Forest
# NO MODEL RETRAINING
# ============================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

folder = "/content/drive/MyDrive/Walmart Recruiting Sales Data"
out_dir = f"{folder}/michaela_replication"
os.makedirs(out_dir, exist_ok=True)

df = combined.copy()
df["Date"] = pd.to_datetime(df["Date"])

# 1. Check required columns
required = [
    "Store", "Date", "Type",
    "ET_MAD_Flag", "IF_Flag",
    "ET_Flagged_Shortfall", "ET_Net_Shortfall"
]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

assert not df.duplicated(["Store", "Date"]).any()

# 2. Find the Isolation Forest anomaly score
# sklearn decision_function: lower = more anomalous
score_candidates = [
    "IF_Anomaly_Score",
    "IF_Score",
    "IsolationForest_Score",
    "IF_Decision_Score"
]

score_col = next(
    (c for c in score_candidates if c in df.columns),
    None
)

assert score_col is not None, (
    "Isolation Forest score not found. "
    "Available columns: " + str(df.columns.tolist())
)

print("Isolation Forest score column:", score_col)

# 3. Attach original store-week markdown indicators
# Avoid relying on department-level markdown assumptions
features = pd.read_csv(
    f"{folder}/features.csv",
    parse_dates=["Date"]
)

markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]

features["Markdown_Recorded_Check"] = (
    features[markdown_cols].notna().any(axis=1)
)
features["Positive_Markdown_Check"] = (
    features[markdown_cols].gt(0).any(axis=1)
)

assert not features.duplicated(["Store", "Date"]).any()

df = df.merge(
    features[
        ["Store", "Date",
         "Markdown_Recorded_Check",
         "Positive_Markdown_Check"]
    ],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# 4. Same business eligibility for BOTH methods
# Current project screening: markdown information recorded
# and aggregate eligible-department sales below forecast
df["ET_Net_Shortfall"] = pd.to_numeric(
    df["ET_Net_Shortfall"], errors="coerce"
)

df["Eligible_Investigation"] = (
    df["Markdown_Recorded_Check"].eq(True)
    & df["ET_Net_Shortfall"].gt(0)
)

# Make sure flags are proper booleans
for col in ["ET_MAD_Flag", "IF_Flag"]:
    df[col] = (
        df[col].astype(str).str.lower()
        .map({"true": True, "false": False,
              "1": True, "0": False})
    )
    assert df[col].notna().all(), f"Invalid {col}"

df[score_col] = pd.to_numeric(
    df[score_col], errors="coerce"
)

# 5. Extra Trees candidate list
# Priority = largest sum of flagged department shortfalls
et_candidates = df[
    df["Eligible_Investigation"] & df["ET_MAD_Flag"]
].copy()

et_candidates = et_candidates.sort_values(
    ["ET_Flagged_Shortfall", "Store", "Date"],
    ascending=[False, True, True]
)

et_top20 = et_candidates.head(20).copy()
et_top20["ET_Rank"] = range(1, len(et_top20) + 1)

# 6. Isolation Forest candidate list
# Only IF-flagged weeks; lower score = more unusual
if_candidates = df[
    df["Eligible_Investigation"]
    & df["IF_Flag"]
    & df[score_col].notna()
].copy()

if_candidates = if_candidates.sort_values(
    [score_col, "Store", "Date"],
    ascending=[True, True, True]
)

if_top20 = if_candidates.head(20).copy()
if_top20["IF_Rank"] = range(1, len(if_top20) + 1)

# 7. Top-20 overlap
et_keys = set(zip(et_top20["Store"], et_top20["Date"]))
if_keys = set(zip(if_top20["Store"], if_top20["Date"]))

intersection = len(et_keys & if_keys)
union = len(et_keys | if_keys)
jaccard_top20 = (
    intersection / union if union else np.nan
)

# 8. Compare both lists using the SAME business metric
# Positive net ET forecast gap over eligible departments
def summarize_top_list(data, name):
    return {
        "Method": name,
        "Cases": len(data),
        "Net_ET_Gap_Sum": data["ET_Net_Shortfall"].sum(),
        "Net_ET_Gap_Median": data["ET_Net_Shortfall"].median(),
        "Type_A": int((data["Type"] == "A").sum()),
        "Type_B": int((data["Type"] == "B").sum()),
        "Type_C": int((data["Type"] == "C").sum())
    }

comparison = pd.DataFrame([
    summarize_top_list(et_top20, "Extra Trees + MAD"),
    summarize_top_list(if_top20, "Isolation Forest")
])

# 9. Build combined priority pool
selected_keys = et_keys | if_keys
priority = df[
    df.apply(
        lambda r: (r["Store"], r["Date"]) in selected_keys,
        axis=1
    )
].copy()

priority["In_ET_Top20"] = [
    (s, d) in et_keys
    for s, d in zip(priority["Store"], priority["Date"])
]

priority["In_IF_Top20"] = [
    (s, d) in if_keys
    for s, d in zip(priority["Store"], priority["Date"])
]

priority["Consensus_Top20"] = (
    priority["In_ET_Top20"] & priority["In_IF_Top20"]
)

priority = priority.sort_values(
    ["Consensus_Top20", "ET_Net_Shortfall"],
    ascending=[False, False]
)

# 10. Display
print("TOP-20 COMPARISON")
print("Eligible store-weeks:", int(df["Eligible_Investigation"].sum()))
print("ET candidates:", len(et_candidates))
print("IF candidates:", len(if_candidates))
print("Cases in both top lists:", intersection)
print("Top-list Jaccard:", round(jaccard_top20, 3))

print("\nBUSINESS IMPACT AND STORE-TYPE COMPARISON")
display(comparison.round(2))

show_cols = [
    "Store", "Date", "Type",
    "ET_MAD_Flag", "IF_Flag",
    "In_ET_Top20", "In_IF_Top20",
    "Consensus_Top20",
    "ET_Net_Shortfall",
    "ET_Flagged_Shortfall",
    score_col
]

print("\nCOMBINED INVESTIGATION CANDIDATES")
display(priority[show_cols].head(40).round(2))

# 11. Save outputs
et_top20.to_csv(
    f"{out_dir}/et_top20_priority.csv",
    index=False
)

if_top20.to_csv(
    f"{out_dir}/if_top20_priority.csv",
    index=False
)

comparison.to_csv(
    f"{out_dir}/top20_business_impact_comparison.csv",
    index=False
)

priority.to_csv(
    f"{out_dir}/combined_top20_priority_pool.csv",
    index=False
)

print("\nSTAGE 12 COMPLETED")
print("Saved to:", out_dir)


Isolation Forest score column: IF_Anomaly_Score
TOP-20 COMPARISON
Eligible store-weeks: 427
ET candidates: 426
IF candidates: 41
Cases in both top lists: 2
Top-list Jaccard: 0.053

BUSINESS IMPACT AND STORE-TYPE COMPARISON


,Method,Cases,Net_ET_Gap_Sum,Net_ET_Gap_Median,Type_A,Type_B,Type_C
0,Extra Trees + MAD,20,2218880.40,112872.93,13,7,0
1,Isolation Forest,20,1044676.19,49064.42,17,3,0



COMBINED INVESTIGATION CANDIDATES


,Store,Date,Type,ET_MAD_Flag,IF_Flag,In_ET_Top20,In_IF_Top20,Consensus_Top20,ET_Net_Shortfall,ET_Flagged_Shortfall,IF_Anomaly_Score
411,28,2012-08-31,A,True,True,True,True,True,113152.17,73933.11,-0.05
50,4,2012-08-24,A,True,True,True,True,True,62565.94,52578.67,-0.03
398,27,2012-09-14,A,True,False,True,False,False,240020.31,99008.93,0.04
188,13,2012-09-14,A,True,False,True,False,False,187831.97,91293.60,0.04
337,23,2012-09-07,B,True,False,True,False,False,170762.84,103512.30,0.01
83,6,2012-09-14,A,True,True,True,False,False,165278.41,53792.16,-0.01
203,14,2012-09-14,A,True,False,True,False,False,159678.31,68832.83,0.06
353,24,2012-09-14,A,True,False,True,False,False,140934.94,55670.06,0.07
256,18,2012-07-27,B,True,False,True,False,False,124642.65,50612.96,0.09
307,21,2012-09-07,B,True,False,True,False,False,121358.43,82161.12,0.03



STAGE 12 COMPLETED
Saved to: /content/drive/MyDrive/Walmart Recruiting Sales Data/michaela_replication


In [9]:

# ============================================================
# STAGE 13 — COMBINED RAG CASES V2
# Extra Trees + Global MAD + Isolation Forest
# NO MODEL RETRAINING
# ============================================================

import os, json, zipfile
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

base = "/content/drive/MyDrive/Walmart Recruiting Sales Data"
rep = f"{base}/michaela_replication"
v2_dir = f"{rep}/rag_cases_v2"
os.makedirs(v2_dir, exist_ok=True)

# 1. Load saved model comparisons and department evidence
all_cases = pd.read_csv(
    f"{rep}/combined_anomaly_comparison.csv",
    parse_dates=["Date"]
)
priority = pd.read_csv(
    f"{rep}/combined_top20_priority_pool.csv",
    parse_dates=["Date"]
)
depts = pd.read_csv(
    f"{base}/et_mad_department_week.csv",
    parse_dates=["Date"]
)
features = pd.read_csv(
    f"{base}/features.csv",
    parse_dates=["Date"]
)

for name, frame in [("combined", all_cases),
                    ("priority", priority)]:
    assert not frame.duplicated(["Store", "Date"]).any(), (
        f"Duplicate Store-Date in {name}"
    )

assert not depts.duplicated(
    ["Store", "Dept", "Date"]
).any()

assert not features.duplicated(
    ["Store", "Date"]
).any()

# 2. Select union of both top-20 lists
#    plus original Store 7 case study
keys = priority[
    ["Store", "Date", "In_ET_Top20", "In_IF_Top20"]
].copy()

for flag in ["In_ET_Top20", "In_IF_Top20"]:
    keys[flag] = (
        keys[flag].astype(str).str.lower()
        .map({"true": True, "false": False})
    )
    assert keys[flag].notna().all()

case_date = pd.Timestamp("2012-08-24")
extra = pd.DataFrame([{
    "Store": 7,
    "Date": case_date,
    "In_ET_Top20": False,
    "In_IF_Top20": False
}])

keys = pd.concat([keys, extra], ignore_index=True)

keys = (
    keys.groupby(["Store", "Date"], as_index=False)
    .agg(
        In_ET_Top20=("In_ET_Top20", "any"),
        In_IF_Top20=("In_IF_Top20", "any")
    )
)

# Include all model results, not only selected rows
selected = keys.merge(
    all_cases,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

assert selected["ET_MAD_Flag"].notna().all(), (
    "Some selected store-weeks lack model results."
)

# 3. Helpers for clean JSON
def clean(value):
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (float, np.number)):
        return float(value) if np.isfinite(value) else None
    if value is None or pd.isna(value):
        return None
    return value

def field(row, name):
    return clean(row.get(name))

def flag(row, name):
    value = row.get(name)
    return str(value).strip().lower() in ("true", "1")

markdown = [f"MarkDown{i}" for i in range(1, 6)]
economic = [
    "Temperature", "Fuel_Price", "CPI", "Unemployment"
]

generated = []

# 4. Generate one JSON for each selected STORE-WEEK
for _, row in selected.iterrows():

    store = int(row["Store"])
    date = pd.Timestamp(row["Date"])
    date_str = date.strftime("%Y-%m-%d")

    dept_rows = depts[
        (depts["Store"] == store) &
        (depts["Date"] == date)
    ]

    context_rows = features[
        (features["Store"] == store) &
        (features["Date"] == date)
    ]
    context = (
        context_rows.iloc[0]
        if len(context_rows) else pd.Series(dtype=object)
    )

    dept_evidence = []

    for _, d in dept_rows.iterrows():
        if flag(d, "Flag_GlobalMAD"):
            dept_evidence.append({
                "department": int(d["Dept"]),
                "actual_sales": field(d, "Actual_Sales"),
                "expected_sales": field(d, "Expected_Sales"),
                "modeled_shortfall": field(d, "Dollar_Shortfall"),
                "shortfall_pct": field(d, "Shortfall_Pct"),
                "global_mad_flag": True
            })

    both_flags = (
        flag(row, "ET_MAD_Flag")
        and flag(row, "IF_Flag")
    )

    selected_by_both = (
        flag(row, "In_ET_Top20")
        and flag(row, "In_IF_Top20")
    )

    case = {
        "case_id": f"store{store}_{date_str}",
        "case_level": "store_week",
        "store": store,
        "date": date_str,
        "store_type": field(row, "Type"),

        "selection": {
            "et_top20": flag(row, "In_ET_Top20"),
            "if_top20": flag(row, "In_IF_Top20"),
            "top20_consensus": selected_by_both,
            "original_store7_case_study": (
                store == 7 and date == case_date
            )
        },

        "supervised": {
            "model": "Extra Trees + Global MAD",
            "store_week_any_department_flag":
                flag(row, "ET_MAD_Flag"),
            "flagged_department_count":
                field(row, "GlobalMAD_Dept_Count"),
            "eligible_department_count":
                field(row, "Eligible_Dept_Count"),
            "net_modeled_shortfall":
                field(row, "ET_Net_Shortfall"),
            "flagged_department_shortfall_sum":
                field(row, "ET_Flagged_Shortfall"),
            "flag_aggregation":
                "At least one eligible department flagged"
        },

        "unsupervised": {
            "model": "Isolation Forest",
            "analysis_level": "store_week",
            "anomaly_flag": flag(row, "IF_Flag"),
            "anomaly_score": next(
                (
                    field(row, c)
                    for c in [
                        "IF_Anomaly_Score",
                        "IF_Score",
                        "IsolationForest_Score",
                        "IF_Decision_Score"
                    ]
                    if c in row.index
                ),
                None
            )
        },

        "model_agreement": {
            "both_flagged": both_flags,
            "interpretation": (
                "Two-model consensus signal"
                if both_flags else
                "Detection methods do not both flag this week"
            )
        },

        "flagged_departments": dept_evidence,

        "external_context": {
            "economic_factors": {
                c: field(context, c) for c in economic
            },
            "markdown_values": {
                c: field(context, c) for c in markdown
            },
            "markdown_recorded": (
                bool(context[markdown].notna().any())
                if len(context) else None
            ),
            "markdown_data_level": "store_week"
        },

        "provenance": {
            "sales_source": "Walmart train.csv",
            "features_source": "Walmart features.csv",
            "supervised_output": "et_mad_department_week.csv",
            "comparison_output": "combined_anomaly_comparison.csv",
            "evaluation_start": "2012-07-20",
            "evaluation_end": "2012-10-26",
            "exported_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        },

        "retrieved_documents": [],
        "retrieval_status": "not_started",

        "evidence_limitations": [
            "Forecast deviations are not confirmed lost revenue.",
            "Markdown values are recorded at store-week level.",
            "Department-specific markdown exposure is unknown.",
            "Isolation Forest anomalies do not necessarily indicate low sales.",
            "Agreement does not prove markdown failure or causality.",
            "An any-department store-week flag is not a calibrated store-level anomaly.",
            "No external documentary root cause has been verified."
        ]
    }

    path = f"{v2_dir}/{case['case_id']}.json"

    with open(path, "w") as f:
        json.dump(case, f, indent=2, allow_nan=False)

    generated.append(case)

# 5. Build summary and ZIP for RAG teammates
summary = pd.DataFrame([
    {
        "Case_ID": x["case_id"],
        "Store": x["store"],
        "Date": x["date"],
        "Type": x["store_type"],
        "ET_Top20": x["selection"]["et_top20"],
        "IF_Top20": x["selection"]["if_top20"],
        "Both_Flagged": x["model_agreement"]["both_flagged"],
        "Net_ET_Shortfall":
            x["supervised"]["net_modeled_shortfall"]
    }
    for x in generated
])

summary.to_csv(f"{rep}/rag_cases_v2_summary.csv", index=False)

zip_path = f"{rep}/walmart_rag_cases_v2.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for case in generated:
        name = case["case_id"] + ".json"
        z.write(f"{v2_dir}/{name}", arcname=name)

print("STAGE 13 COMPLETE")
print("Store-week JSON cases:", len(generated))
print("Both top-20:", int(
    (summary["ET_Top20"] & summary["IF_Top20"]).sum()
))
print("ZIP:", zip_path)

display(summary.head(15))


STAGE 13 COMPLETE
Store-week JSON cases: 39
Both top-20: 2
ZIP: /content/drive/MyDrive/Walmart Recruiting Sales Data/michaela_replication/walmart_rag_cases_v2.zip


,Case_ID,Store,Date,Type,ET_Top20,IF_Top20,Both_Flagged,Net_ET_Shortfall
0,store2_2012-09-07,2,2012-09-07,A,True,False,False,53017.154776
1,store4_2012-07-27,4,2012-07-27,A,False,True,True,112580.797483
2,store4_2012-08-03,4,2012-08-03,A,False,True,True,46637.280617
3,store4_2012-08-24,4,2012-08-24,A,True,True,True,62565.935844
4,store4_2012-08-31,4,2012-08-31,A,False,True,True,88738.142356
5,store4_2012-09-07,4,2012-09-07,A,False,True,True,53619.196883
6,store4_2012-09-14,4,2012-09-14,A,False,True,True,13283.759183
7,store6_2012-08-03,6,2012-08-03,A,False,True,True,48798.392889
8,store6_2012-08-31,6,2012-08-31,A,False,True,True,21566.760389
9,store6_2012-09-14,6,2012-09-14,A,True,False,True,165278.405639


**Latest Update:**

In [10]:

import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")

et = pd.read_csv(base / "et_mad_store_week.csv")
michaela = pd.read_csv(
    base / "michaela_replication" /
    "michaela_store_week_comparison.csv"
)

et["Date"] = pd.to_datetime(et["Date"])
michaela["Date"] = pd.to_datetime(michaela["Date"])

# Keep both methods' existing flags and scores
combined_updated = et.merge(
    michaela[[
        "Store", "Date", "IF_Flag",
        "IF_Anomaly_Score",
        "Store_MA4", "Store_MA8",
        "MA_TypeMAD_Any"
    ]],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

assert len(combined_updated) == len(et)
assert combined_updated["IF_Flag"].notna().all()

# How frequently do methods agree?
combined_updated["Both_Flag"] = (
    combined_updated["ET_MAD_Flag"].astype(bool)
    & combined_updated["IF_Flag"].astype(bool)
)

rows = []
for method, column in [
    ("Extra Trees + Global MAD", "ET_MAD_Flag"),
    ("Chronological Isolation Forest", "IF_Flag"),
    ("Both Methods", "Both_Flag"),
    ("Moving Average + Type MAD", "MA_TypeMAD_Any")
]:
    selected = combined_updated[
        combined_updated[column].astype(bool)
    ]
    rows.append({
        "Method": method,
        "Flagged_Store_Weeks": len(selected),
        "Flag_Rate_Pct": round(
            100 * len(selected) / len(combined_updated), 2
        ),
        "ET_Net_Modeled_Shortfall": round(
            selected["ET_Net_Shortfall"].sum(), 2
        )
    })

evaluation_matrix = pd.DataFrame(rows)

display(evaluation_matrix)

combined_updated.to_csv(
    base / "combined_updated_with_benchmarks.csv",
    index=False
)

evaluation_matrix.to_csv(
    base / "combined_evaluation_matrix.csv",
    index=False
)

print("Saved both comparison files to Google Drive.")


,Method,Flagged_Store_Weeks,Flag_Rate_Pct,ET_Net_Modeled_Shortfall
0,Extra Trees + Global MAD,672,99.56,15298478.79
1,Chronological Isolation Forest,68,10.07,2266330.33
2,Both Methods,67,9.93,2254729.58
3,Moving Average + Type MAD,554,82.07,14799643.96


Saved both comparison files to Google Drive.


In [11]:

# ==========================================================
# FINAL HYBRID: ISOLATION FOREST -> EXTRA TREES + GLOBAL MAD
# NO RETRAINING
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

base = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
replica = base / "michaela_replication"

# 1. Load our already-saved results
et = pd.read_csv(base / "et_mad_store_week_with_factors.csv")
iso = pd.read_csv(replica / "michaela_store_week_comparison.csv")
dept = pd.read_csv(base / "et_mad_department_week.csv")

for df in [et, iso, dept]:
    df["Date"] = pd.to_datetime(df["Date"])

# Convert stored CSV flags safely to Boolean
def as_bool(s):
    result = (
        s.astype(str).str.strip().str.lower()
        .map({"true": True, "false": False,
              "1": True, "0": False})
    )
    assert result.notna().all(), f"Invalid flags: {s.name}"
    return result

et["ET_MAD_Flag"] = as_bool(et["ET_MAD_Flag"])
iso["IF_Flag"] = as_bool(iso["IF_Flag"])
dept["Flag_GlobalMAD"] = as_bool(dept["Flag_GlobalMAD"])

# 2. Match the two models on Store and Date
assert not et.duplicated(["Store", "Date"]).any()
assert not iso.duplicated(["Store", "Date"]).any()

hybrid = et.merge(
    iso[["Store", "Date", "IF_Flag",
         "IF_Anomaly_Score"]],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

assert len(hybrid) == 675, "Unexpected evaluation row count"
assert hybrid["IF_Flag"].notna().all(), "Missing IF predictions"

# 3. Apply the final screening rules
hybrid["Hybrid_Flag"] = (
    hybrid["IF_Flag"]
    & hybrid["ET_MAD_Flag"]
    & (hybrid["ET_Net_Shortfall"] > 0)
    & hybrid["Markdown_Recorded"].astype(bool)
)

# 4. Select all qualifying cases
cases = hybrid.loc[hybrid["Hybrid_Flag"]].copy()

# Higher modeled shortfall = higher investigation priority
cases = cases.sort_values(
    ["ET_Flagged_Shortfall", "IF_Anomaly_Score"],
    ascending=[False, True]
).reset_index(drop=True)

cases.insert(0, "Priority_Rank",
             range(1, len(cases) + 1))

cases["Case_ID"] = (
    "Store_" + cases["Store"].astype(str)
    + "_" + cases["Date"].dt.strftime("%Y-%m-%d")
)

# 5. Get flagged departments for selected store-weeks
flagged_depts = dept.loc[dept["Flag_GlobalMAD"]].copy()

case_keys = cases[["Store", "Date", "Case_ID", "Priority_Rank"]]

case_departments = flagged_depts.merge(
    case_keys,
    on=["Store", "Date"],
    how="inner",
    validate="many_to_one"
)

case_departments = case_departments.sort_values(
    ["Priority_Rank", "Dollar_Shortfall"],
    ascending=[True, False]
)

# 6. Export ALL cases and Top 20
cases.to_csv(
    base / "FINAL_hybrid_all_cases.csv", index=False
)

cases.head(20).to_csv(
    base / "FINAL_hybrid_top20_cases.csv", index=False
)

case_departments.to_csv(
    base / "FINAL_hybrid_flagged_departments.csv",
    index=False
)

# 7. Display results
print("======================================")
print("FINAL HYBRID RESULTS")
print("======================================")
print("Total evaluated:", len(hybrid))
print("Isolation Forest flags:", int(hybrid["IF_Flag"].sum()))
print("Both model flags:", int(
    (hybrid["IF_Flag"] & hybrid["ET_MAD_Flag"]).sum()
))
print("Final qualifying cases:", len(cases))
print("Stores represented:", cases["Store"].nunique())
print("Final cases by store type:")
print(cases["Type"].value_counts().sort_index())

print("\nTop 20 investigation cases:")

display(cases[[
    "Priority_Rank", "Store", "Date", "Type",
    "ET_Net_Shortfall", "ET_Flagged_Shortfall",
    "GlobalMAD_Dept_Count", "IF_Anomaly_Score"
]].head(20))

print("\nSaved 3 new CSVs to Google Drive.")


FINAL HYBRID RESULTS
Total evaluated: 675
Isolation Forest flags: 68
Both model flags: 67
Final qualifying cases: 40
Stores represented: 18
Final cases by store type:
Type
A    30
B    10
Name: count, dtype: int64

Top 20 investigation cases:


,Priority_Rank,Store,Date,Type,ET_Net_Shortfall,ET_Flagged_Shortfall,GlobalMAD_Dept_Count,IF_Anomaly_Score
0,1,39,2012-09-07,A,112593.702039,77423.680672,10,-0.009816
1,2,28,2012-08-31,A,113152.165367,73933.112367,11,-0.048670
2,3,12,2012-09-07,B,79936.806244,60019.557928,11,-0.020786
3,4,6,2012-09-14,A,165278.405639,53792.159111,14,-0.010174
4,5,4,2012-08-24,A,62565.935844,52578.667539,12,-0.032452
5,6,8,2012-09-07,A,36413.000556,48961.157411,14,-0.001548
6,7,10,2012-08-10,B,18740.906167,44248.942300,9,-0.012452
7,8,10,2012-08-31,B,156812.001294,41342.337156,7,-0.007617
8,9,4,2012-08-31,A,88738.142356,32905.655878,7,-0.039113
9,10,4,2012-09-07,A,53619.196883,30729.931478,16,-0.038782



Saved 3 new CSVs to Google Drive.


**V3.** Changed from the union of separate Top-20 lists to Isolation Forest first, followed by Extra Trees + Global MAD. Selected 40 cases from 675 store-weeks.

In [12]:

# ============================================================
# STAGE 15 — V3 RAG JSON EXPORT
# Hybrid: Chronological Isolation Forest -> Extra Trees MAD
# Creates one JSON per case + summary + ZIP
# NO MODEL RETRAINING
# ============================================================

import pandas as pd
import numpy as np
import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
OUT = BASE / "rag_cases_v3"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD FINAL HYBRID CASES AND DEPARTMENT EVIDENCE
# ------------------------------------------------------------

cases = pd.read_csv(BASE / "FINAL_hybrid_all_cases.csv")
depts = pd.read_csv(BASE / "FINAL_hybrid_flagged_departments.csv")

cases["Date"] = pd.to_datetime(cases["Date"])
depts["Date"] = pd.to_datetime(depts["Date"])

assert len(cases) == 40, (
    f"Expected 40 hybrid cases, found {len(cases)}"
)
assert not cases.duplicated(["Store", "Date"]).any()
assert cases["Priority_Rank"].is_unique

print(f"Loaded {len(cases)} final hybrid cases.")
print(f"Loaded {len(depts)} flagged department records.")

# ------------------------------------------------------------
# 2. SAFE VALUE CONVERSION
# ------------------------------------------------------------

def clean_value(value):
    """Convert pandas/numpy values into valid JSON-safe values."""
    if value is None:
        return None

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.strftime("%Y-%m-%d")

    if isinstance(value, (np.bool_, bool)):
        return bool(value)

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None

    if isinstance(value, (np.ndarray, list, tuple)):
        return [clean_value(x) for x in value]

    if isinstance(value, dict):
        return {str(k): clean_value(v) for k, v in value.items()}

    if pd.isna(value):
        return None

    return value


def get(row, column, default=None):
    if column not in row.index:
        return default
    return clean_value(row[column])


def numeric(row, column):
    value = get(row, column)
    return float(value) if value is not None else None


def record_bool(value):
    return str(value).strip().lower() in ("true", "1")


# ------------------------------------------------------------
# 3. CASE-LEVEL HELPER
# ------------------------------------------------------------

def build_case(row):

    store = int(row["Store"])
    date = pd.Timestamp(row["Date"])
    date_str = date.strftime("%Y-%m-%d")
    rank = int(row["Priority_Rank"])

    case_id = f"Store_{store}_{date_str}"

    subset = depts.loc[
        (depts["Store"] == store) &
        (depts["Date"] == date)
    ].copy()

    # Sort by largest positive department modeled gap
    if "Dollar_Shortfall" in subset.columns:
        subset = subset.sort_values(
            "Dollar_Shortfall", ascending=False
        )

    department_evidence = []

    for _, d in subset.iterrows():
        department_evidence.append({
            "department_id": get(d, "Dept"),
            "actual_weekly_sales": get(d, "Weekly_Sales"),
            "expected_weekly_sales_extra_trees": get(
                d, "Expected_Sales"
            ),
            "modeled_dollar_shortfall": get(
                d, "Dollar_Shortfall"
            ),
            "modeled_shortfall_percent": get(
                d, "Shortfall_Pct"
            ),
            "global_mad_flag": get(
                d, "Flag_GlobalMAD"
            )
        })

    markdown_columns = [
        "MarkDown1", "MarkDown2", "MarkDown3",
        "MarkDown4", "MarkDown5"
    ]

    markdown_data = {
        col: get(row, col)
        for col in markdown_columns
    }

    payload = {
        "schema_version": "3.0",
        "case_id": case_id,
        "source": {
            "project": "Walmart Markdown Exception Detection",
            "dataset": "Walmart Recruiting Store Sales Forecasting",
            "evaluation_period": {
                "start": "2012-07-20",
                "end": "2012-10-26"
            },
            "pipeline": (
                "Chronological Isolation Forest screening "
                "followed by Extra Trees Global MAD "
                "underperformance verification"
            ),
            "source_files": [
                "FINAL_hybrid_all_cases.csv",
                "FINAL_hybrid_flagged_departments.csv"
            ]
        },
        "store_week": {
            "store_id": store,
            "date": date_str,
            "store_type": get(row, "Type"),
            "priority_rank": rank
        },
        "hybrid_detection": {
            "final_hybrid_flag": get(row, "Hybrid_Flag"),
            "isolation_forest": {
                "flag": get(row, "IF_Flag"),
                "anomaly_score": get(
                    row, "IF_Anomaly_Score"
                ),
                "interpretation": (
                    "Store-week exhibits an unusual pattern "
                    "relative to the chronological Isolation "
                    "Forest training distribution. This does "
                    "not by itself establish low sales."
                )
            },
            "extra_trees_global_mad": {
                "store_week_flag": get(row, "ET_MAD_Flag"),
                "flagged_department_count": get(
                    row, "GlobalMAD_Dept_Count"
                ),
                "modeled_net_shortfall": get(
                    row, "ET_Net_Shortfall"
                ),
                "sum_flagged_department_shortfalls": get(
                    row, "ET_Flagged_Shortfall"
                ),
                "interpretation": (
                    "Department-level actual sales are "
                    "compared against Extra Trees forecasts. "
                    "Global MAD flags unusually negative "
                    "forecast residuals."
                )
            },
            "selection_rules": {
                "isolation_forest_flag_required": True,
                "extra_trees_global_mad_flag_required": True,
                "positive_aggregate_et_gap_required": True,
                "markdown_recorded_required": True,
                "ranking_metric": (
                    "ET_Flagged_Shortfall descending, "
                    "IF_Anomaly_Score ascending"
                )
            }
        },
        "sales_evidence": {
            "extra_trees_eligible_expected_sales": get(
                row, "ET_Expected_Sales"
            ),
            "extra_trees_eligible_actual_sales": get(
                row, "ET_Actual_Sales"
            ),
            "net_modeled_shortfall": get(
                row, "ET_Net_Shortfall"
            ),
            "flagged_departments": department_evidence
        },
        "markdown_context": {
            "markdown_recorded": get(
                row, "Markdown_Recorded"
            ),
            "recorded_markdown_count": get(
                row, "Recorded_Markdown_Count"
            ),
            "recorded_markdown_total": get(
                row, "Recorded_Markdown_Total"
            ),
            "markdown_fields": markdown_data,
            "interpretation": (
                "Markdown variables are recorded at the "
                "store-week level, not department level. "
                "Missing values must not automatically "
                "be interpreted as zero markdown activity."
            )
        },
        "external_context": {
            "temperature": get(row, "Temperature"),
            "fuel_price": get(row, "Fuel_Price"),
            "cpi": get(row, "CPI"),
            "unemployment": get(row, "Unemployment"),
            "interpretation": (
                "These are contextual observations, "
                "not validated causal explanations."
            )
        },
        "rag_investigation": {
            "retrieved_documents": [],
            "questions_for_agent": [
                "Which flagged departments account for "
                "the largest modeled sales gaps?",
                "Does available historical evidence support "
                "or challenge the apparent underperformance?",
                "Are markdown records or external conditions "
                "consistent with any plausible explanations?",
                "What additional evidence would be required "
                "before recommending a business action?"
            ],
            "limitations": [
                "No verified ground-truth anomaly labels.",
                "Forecast residuals are not proof of causality.",
                "Modeled dollar shortfalls are not confirmed "
                "lost revenue.",
                "Store-week markdown records do not establish "
                "department-level markdown activity.",
                "Isolation Forest detects atypical patterns "
                "and does not inherently determine whether "
                "sales are below expectations.",
                "Missing markdown values do not necessarily "
                "indicate no markdown activity.",
                "Results are restricted to the final test "
                "period and available model features."
            ],
            "recommended_action_status": (
                "Investigate — no root cause confirmed"
            )
        }
    }

    return clean_value(payload)


# ------------------------------------------------------------
# 4. GENERATE ONE JSON PER STORE-WEEK
# ------------------------------------------------------------

written = []

for _, row in cases.sort_values("Priority_Rank").iterrows():

    payload = build_case(row)

    rank = int(row["Priority_Rank"])
    case_id = payload["case_id"]

    filename = f"rank_{rank:02d}_{case_id}.json"
    filepath = OUT / filename

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(
            payload, f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )

    written.append(filepath)

assert len(written) == 40

print(f"Generated {len(written)} V3 JSON files.")

# ------------------------------------------------------------
# 5. CREATE V3 SUMMARY CSV
# ------------------------------------------------------------

summary_cols = [
    "Priority_Rank",
    "Case_ID",
    "Store",
    "Date",
    "Type",
    "IF_Anomaly_Score",
    "ET_Net_Shortfall",
    "ET_Flagged_Shortfall",
    "GlobalMAD_Dept_Count",
    "Markdown_Recorded",
    "Recorded_Markdown_Total"
]

available_summary_cols = [
    col for col in summary_cols if col in cases.columns
]

summary = cases[available_summary_cols].copy()

summary.to_csv(
    BASE / "rag_cases_v3_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 6. ZIP ALL V3 JSON FILES
# ------------------------------------------------------------

zip_path = BASE / "walmart_rag_cases_v3.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zf:
    for filepath in written:
        zf.write(
            filepath,
            arcname=f"rag_cases_v3/{filepath.name}"
        )

# ------------------------------------------------------------
# 7. FINAL VERIFICATION
# ------------------------------------------------------------

with zipfile.ZipFile(zip_path) as zf:
    json_files = [
        x for x in zf.namelist()
        if x.endswith(".json")
    ]

    assert len(json_files) == 40

    # Test all JSONs can be loaded
    for name in json_files:
        json.loads(zf.read(name))

print("\n===================================")
print("V3 RAG EXPORT COMPLETE")
print("===================================")
print("JSON cases:", len(written))
print("Stores represented:", cases["Store"].nunique())
print("Store types:")
print(cases["Type"].value_counts().sort_index())
print("V3 JSON folder:", OUT)
print("Summary:", BASE / "rag_cases_v3_summary.csv")
print("ZIP:", zip_path)

print("\nTop 5 cases:")
display(summary.head(5))


Loaded 40 final hybrid cases.
Loaded 302 flagged department records.
Generated 40 V3 JSON files.

V3 RAG EXPORT COMPLETE
JSON cases: 40
Stores represented: 18
Store types:
Type
A    30
B    10
Name: count, dtype: int64
V3 JSON folder: /content/drive/MyDrive/Walmart Recruiting Sales Data/rag_cases_v3
Summary: /content/drive/MyDrive/Walmart Recruiting Sales Data/rag_cases_v3_summary.csv
ZIP: /content/drive/MyDrive/Walmart Recruiting Sales Data/walmart_rag_cases_v3.zip

Top 5 cases:


,Priority_Rank,Case_ID,Store,Date,Type,IF_Anomaly_Score,ET_Net_Shortfall,ET_Flagged_Shortfall,GlobalMAD_Dept_Count,Markdown_Recorded,Recorded_Markdown_Total
0,1,Store_39_2012-09-07,39,2012-09-07,A,-0.009816,112593.702039,77423.680672,10,True,35328.97
1,2,Store_28_2012-08-31,28,2012-08-31,A,-0.048670,113152.165367,73933.112367,11,True,42230.37
2,3,Store_12_2012-09-07,12,2012-09-07,B,-0.020786,79936.806244,60019.557928,11,True,23606.58
3,4,Store_6_2012-09-14,6,2012-09-14,A,-0.010174,165278.405639,53792.159111,14,True,24970.27
4,5,Store_4_2012-08-24,4,2012-08-24,A,-0.032452,62565.935844,52578.667539,12,True,25281.54


**V3.1 - Evidence completeness:** Fixed missing actual-sales fields, clarified aggregate versus flagged-department dollar shortfalls, added historical predictions, and checked numerical reconciliation.

In [13]:

# ==========================================================
# V3.1 RAG EXPORT — IMPROVED HYBRID INVESTIGATION EVIDENCE
# 40 cases | No retraining | Preserves V1, V2 and V3
# ==========================================================

import pandas as pd
import numpy as np
import json
import zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
OUT = BASE / "rag_cases_v3_1"
OUT.mkdir(parents=True, exist_ok=True)

cases = pd.read_csv(BASE / "FINAL_hybrid_all_cases.csv")
depts = pd.read_csv(BASE / "FINAL_hybrid_flagged_departments.csv")

cases["Date"] = pd.to_datetime(cases["Date"])
depts["Date"] = pd.to_datetime(depts["Date"])

assert len(cases) == 40, "Expected 40 hybrid cases."
assert not cases.duplicated(["Store", "Date"]).any()
assert not cases["Priority_Rank"].duplicated().any()

# ----------------------------------------------------------
# 1. IDENTIFY ORIGINAL SALES COLUMNS
# ----------------------------------------------------------

def find_col(df, candidates):
    lookup = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

DEPT_ACTUAL = find_col(depts, [
    "Actual_Sales", "Weekly_Sales", "Actual_Weekly_Sales",
    "Actual", "y_true"
])

DEPT_EXPECTED = find_col(depts, [
    "Expected_Sales", "Predicted_Sales", "ET_Predicted_Sales",
    "ET_Expected_Sales"
])

DEPT_SHORTFALL = find_col(depts, [
    "Dollar_Shortfall"
])

STORE_ACTUAL = find_col(cases, [
    "ET_Actual_Eligible", "ET_Eligible_Actual",
    "ET_Actual_Sales", "ET_Actual_Eligible_Sales",
    "ET_Actual_Sum"
])

STORE_EXPECTED = find_col(cases, [
    "ET_Expected_Eligible", "ET_Eligible_Expected",
    "ET_Expected_Sales", "ET_Expected_Eligible_Sales",
    "ET_Expected_Sum"
])

print("Department actual column:", DEPT_ACTUAL)
print("Department expected column:", DEPT_EXPECTED)
print("Store actual column:", STORE_ACTUAL)
print("Store expected column:", STORE_EXPECTED)

# ----------------------------------------------------------
# 2. JSON-SAFE CONVERSION
# ----------------------------------------------------------

def safe(v):
    if v is None:
        return None
    if isinstance(v, (pd.Timestamp, datetime)):
        return v.strftime("%Y-%m-%d")
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return float(v) if np.isfinite(v) else None
    if isinstance(v, (list, tuple, np.ndarray)):
        return [safe(x) for x in v]
    if isinstance(v, dict):
        return {str(k): safe(x) for k, x in v.items()}
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v

def val(row, key):
    if key is None or key not in row.index:
        return None
    return safe(row[key])

def number(row, key):
    v = val(row, key)
    return float(v) if v is not None else None

def flagged(v):
    return str(v).strip().lower() in ("true", "1")

# ----------------------------------------------------------
# 3. LOAD OPTIONAL HISTORICAL PREDICTIONS
# ----------------------------------------------------------

history_path = BASE / "historical_et_predictions.csv"
history = None

if history_path.exists():
    candidate = pd.read_csv(history_path)

    h_store = find_col(candidate, ["Store"])
    h_dept = find_col(candidate, ["Dept"])
    h_date = find_col(candidate, ["Date"])
    h_actual = find_col(candidate, [
        "Actual_Sales", "Weekly_Sales", "Actual"
    ])
    h_expected = find_col(candidate, [
        "Expected_Sales", "Predicted_Sales",
        "ET_Predicted_Sales"
    ])

    if all([h_store, h_dept, h_date, h_actual, h_expected]):
        candidate[h_date] = pd.to_datetime(
            candidate[h_date], errors="coerce"
        )
        history = candidate
        print("Historical prediction evidence available.")
    else:
        print(
            "Historical file found, but required columns "
            "are unavailable. History will not be inferred."
        )
else:
    print("No historical prediction file found; skipping history.")

def get_history(store, dept, case_date):
    if history is None:
        return []

    subset = history.loc[
        (history[h_store] == store) &
        (history[h_dept] == dept) &
        (history[h_date] < case_date)
    ].sort_values(h_date).tail(4)

    return [
        {
            "date": safe(r[h_date]),
            "actual_sales": safe(r[h_actual]),
            "expected_sales": safe(r[h_expected])
        }
        for _, r in subset.iterrows()
    ]

# ----------------------------------------------------------
# 4. BUILD EACH CASE
# ----------------------------------------------------------

def build_case(row):
    store = int(row["Store"])
    date = pd.Timestamp(row["Date"])
    date_str = date.strftime("%Y-%m-%d")
    rank = int(row["Priority_Rank"])
    case_id = f"Store_{store}_{date_str}"

    selected = depts.loc[
        (depts["Store"] == store) &
        (depts["Date"] == date)
    ].copy()

    selected = selected.sort_values(
        "Dollar_Shortfall", ascending=False
    )

    department_evidence = []

    for _, d in selected.iterrows():
        expected = number(d, DEPT_EXPECTED)
        actual = number(d, DEPT_ACTUAL)
        gap = number(d, DEPT_SHORTFALL)

        source = (
            f"Observed field: {DEPT_ACTUAL}"
            if actual is not None else
            "Not available in department export"
        )

        # Reconstruct only when the required values exist.
        # This is explicitly identified as derived, not observed.
        if actual is None and expected is not None and gap is not None:
            actual = expected - gap
            source = (
                "Derived: expected sales minus positive "
                "modeled dollar shortfall"
            )

        dept_id = int(d["Dept"])

        department_evidence.append({
            "department_id": dept_id,
            "actual_weekly_sales": actual,
            "actual_sales_source": source,
            "expected_weekly_sales_extra_trees": expected,
            "modeled_dollar_shortfall": gap,
            "modeled_shortfall_percent": val(
                d, "Shortfall_Pct"
            ),
            "global_mad_flag": flagged(
                val(d, "Flag_GlobalMAD")
            ),
            "historical_predictions": get_history(
                store, dept_id, date
            )
        })

    flagged_total = number(row, "ET_Flagged_Shortfall")
    net_total = number(row, "ET_Net_Shortfall")

    # Directly supplied aggregate values, if available
    store_actual = number(row, STORE_ACTUAL)
    store_expected = number(row, STORE_EXPECTED)

    # Check that the flagged-department detail reconciles
    detail_gaps = [
        d["modeled_dollar_shortfall"]
        for d in department_evidence
        if d["modeled_dollar_shortfall"] is not None
    ]

    detail_sum = sum(detail_gaps)

    difference = (
        abs(detail_sum - flagged_total)
        if flagged_total is not None else None
    )

    reconciles = (
        difference <= max(0.05, 1e-6 * abs(flagged_total))
        if difference is not None else None
    )

    payload = {
        "schema_version": "3.1",
        "case_id": case_id,

        "source": {
            "project": "Walmart Markdown Exception Detection",
            "pipeline": (
                "Chronological Isolation Forest screening "
                "followed by Extra Trees Global MAD "
                "sales-underperformance filtering"
            ),
            "evaluation_period": {
                "start": "2012-07-20",
                "end": "2012-10-26"
            },
            "input_files": [
                "FINAL_hybrid_all_cases.csv",
                "FINAL_hybrid_flagged_departments.csv"
            ],
            "historical_file_used": (
                history_path.name if history is not None
                else None
            ),
            "retraining_performed": False
        },

        "store_week": {
            "store_id": store,
            "date": date_str,
            "store_type": val(row, "Type"),
            "priority_rank": rank
        },

        "hybrid_detection": {
            "final_hybrid_flag": flagged(
                val(row, "Hybrid_Flag")
            ),
            "isolation_forest": {
                "flag": flagged(val(row, "IF_Flag")),
                "anomaly_score": val(
                    row, "IF_Anomaly_Score"
                ),
                "interpretation": (
                    "An unusual multivariate pattern, "
                    "not necessarily a negative sales event."
                )
            },
            "extra_trees_global_mad": {
                "flag": flagged(
                    val(row, "ET_MAD_Flag")
                ),
                "flagged_department_count": len(
                    department_evidence
                ),
                "interpretation": (
                    "Flags unusually negative "
                    "department-level forecast residuals."
                )
            },
            "selection_rules": {
                "isolation_forest_flag_required": True,
                "global_mad_department_flag_required": True,
                "positive_aggregate_et_gap_required": True,
                "markdown_recorded_required": True,
                "priority_metric": (
                    "Sum of flagged department modeled "
                    "shortfalls, descending"
                ),
                "tie_breaker": (
                    "Isolation Forest anomaly score, ascending"
                )
            }
        },

        "sales_evidence": {
            "eligible_department_aggregate": {
                "expected_sales": store_expected,
                "actual_sales": store_actual,
                "net_modeled_shortfall": net_total,
                "net_shortfall_definition": (
                    "Maximum of zero and the difference "
                    "between summed eligible expected "
                    "and actual sales."
                )
            },
            "global_mad_flagged_departments": {
                "count": len(department_evidence),
                "sum_positive_modeled_shortfalls": flagged_total,
                "detail_sum": round(detail_sum, 6),
                "detail_matches_aggregate": reconciles,
                "departments": department_evidence
            },
            "important_distinction": (
                "Aggregate net shortfall includes all "
                "eligible departments, whereas flagged "
                "department shortfall includes only "
                "Global MAD-flagged departments. "
                "Neither represents confirmed lost revenue."
            )
        },

        "markdown_context": {
            "markdown_recorded": flagged(
                val(row, "Markdown_Recorded")
            ),
            "recorded_markdown_count": val(
                row, "Recorded_Markdown_Count"
            ),
            "recorded_markdown_total": val(
                row, "Recorded_Markdown_Total"
            ),
            "markdown_fields": {
                f"MarkDown{i}": val(row, f"MarkDown{i}")
                for i in range(1, 6)
            },
            "interpretation": (
                "Markdowns are store-week variables. "
                "Recorded amounts do not prove that "
                "any particular department was discounted. "
                "Missing is not equivalent to zero."
            )
        },

        "external_context": {
            "temperature": val(row, "Temperature"),
            "fuel_price": val(row, "Fuel_Price"),
            "cpi": val(row, "CPI"),
            "unemployment": val(row, "Unemployment"),
            "interpretation": (
                "Contextual observations only; "
                "no causal attribution established."
            )
        },

        "data_quality": {
            "department_actual_values_available": all(
                d["actual_weekly_sales"] is not None
                for d in department_evidence
            ),
            "direct_store_aggregate_actual_available": (
                store_actual is not None
            ),
            "direct_store_aggregate_expected_available": (
                store_expected is not None
            ),
            "flagged_shortfall_reconciles": reconciles,
            "historical_evidence_available": any(
                d["historical_predictions"]
                for d in department_evidence
            ),
            "notes": (
                "Null means not available from the "
                "validated source columns. Derived values "
                "are identified by their source."
            )
        },

        "rag_investigation": {
            "retrieved_documents": [],
            "questions_for_agent": [
                "Which flagged departments have the "
                "largest modeled negative sales deviations?",
                "Do historical observations support "
                "or challenge these deviations?",
                "What markdown and external context "
                "is available?",
                "Which explanations require additional "
                "operational or external evidence?",
                "What is the safest next investigation step?"
            ],
            "guardrails": [
                "Do not claim markdown causality.",
                "Do not treat modeled shortfalls as "
                "verified revenue losses.",
                "Do not infer department-level markdown "
                "activity from store-level fields.",
                "Do not equate anomaly scores with "
                "probabilities.",
                "Distinguish actual observations, "
                "derived fields and model estimates.",
                "Do not fabricate historical comparisons "
                "or retrieved documents.",
                "No verified anomaly ground-truth labels."
            ],
            "recommended_action_status": (
                "Requires investigation; root cause unknown"
            )
        }
    }

    return safe(payload)

# ----------------------------------------------------------
# 5. SAVE THE 40 NEW JSON FILES
# ----------------------------------------------------------

written = []

for _, row in cases.sort_values("Priority_Rank").iterrows():
    payload = build_case(row)

    filename = (
        f"rank_{int(row['Priority_Rank']):02d}_"
        f"{payload['case_id']}.json"
    )

    path = OUT / filename

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload, f, indent=2,
            ensure_ascii=False, allow_nan=False
        )

    written.append(path)

# ----------------------------------------------------------
# 6. SUMMARY + ZIP + VALIDATION
# ----------------------------------------------------------

summary_rows = []

for path in written:
    with open(path, encoding="utf-8") as f:
        p = json.load(f)

    se = p["sales_evidence"]

    summary_rows.append({
        "Case_ID": p["case_id"],
        "Priority_Rank": p["store_week"]["priority_rank"],
        "Store": p["store_week"]["store_id"],
        "Date": p["store_week"]["date"],
        "Type": p["store_week"]["store_type"],
        "ET_Net_Shortfall":
            se["eligible_department_aggregate"][
                "net_modeled_shortfall"
            ],
        "ET_Flagged_Shortfall":
            se["global_mad_flagged_departments"][
                "sum_positive_modeled_shortfalls"
            ],
        "Flagged_Department_Count":
            se["global_mad_flagged_departments"]["count"],
        "History_Available":
            p["data_quality"][
                "historical_evidence_available"
            ],
        "Shortfall_Reconciles":
            p["data_quality"][
                "flagged_shortfall_reconciles"
            ]
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(
    BASE / "rag_cases_v3_1_summary.csv", index=False
)

zip_path = BASE / "walmart_rag_cases_v3_1.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zf:
    for path in written:
        zf.write(
            path, arcname=f"rag_cases_v3_1/{path.name}"
        )

with zipfile.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist()
             if n.endswith(".json")]
    assert len(names) == len(cases) == 40

    for name in names:
        json.loads(zf.read(name))

print("\n==================================")
print("V3.1 RAG EXPORT COMPLETE")
print("==================================")
print("JSON cases:", len(written))
print("Stores:", cases["Store"].nunique())
print("Shortfall reconciliation failures:",
      int((summary["Shortfall_Reconciles"] == False).sum()))
print("Cases with historical evidence:",
      int(summary["History_Available"].sum()))
print("ZIP:", zip_path)

display(summary.head(10))


Department actual column: Actual_Sales
Department expected column: Expected_Sales
Store actual column: ET_Actual_Eligible_Sales
Store expected column: ET_Expected_Eligible_Sales
Historical prediction evidence available.

V3.1 RAG EXPORT COMPLETE
JSON cases: 40
Stores: 18
Shortfall reconciliation failures: 0
Cases with historical evidence: 40
ZIP: /content/drive/MyDrive/Walmart Recruiting Sales Data/walmart_rag_cases_v3_1.zip


,Case_ID,Priority_Rank,Store,Date,Type,ET_Net_Shortfall,ET_Flagged_Shortfall,Flagged_Department_Count,History_Available,Shortfall_Reconciles
0,Store_39_2012-09-07,1,39,2012-09-07,A,112593.702039,77423.680672,10,True,True
1,Store_28_2012-08-31,2,28,2012-08-31,A,113152.165367,73933.112367,11,True,True
2,Store_12_2012-09-07,3,12,2012-09-07,B,79936.806244,60019.557928,11,True,True
3,Store_6_2012-09-14,4,6,2012-09-14,A,165278.405639,53792.159111,14,True,True
4,Store_4_2012-08-24,5,4,2012-08-24,A,62565.935844,52578.667539,12,True,True
5,Store_8_2012-09-07,6,8,2012-09-07,A,36413.000556,48961.157411,14,True,True
6,Store_10_2012-08-10,7,10,2012-08-10,B,18740.906167,44248.942300,9,True,True
7,Store_10_2012-08-31,8,10,2012-08-31,B,156812.001294,41342.337156,7,True,True
8,Store_4_2012-08-31,9,4,2012-08-31,A,88738.142356,32905.655878,7,True,True
9,Store_4_2012-09-07,10,4,2012-09-07,A,53619.196883,30729.931478,16,True,True


**V3.2 - Historical and root-cause validation**: Added recent 4/8-week sales comparisons, previous-year seasonal benchmarks, cross-store comparisons, external-factor comparisons, and Added recent 4/8-week sales comparisons, previous-year seasonal benchmarks, cross-store comparisons, external-factor comparisons, and structured root-cause hypotheses.

In [14]:

# ============================================================
# STAGE 16 — V3.2 HISTORICAL VALIDATION + ROOT-CAUSE REVIEW
# Reuses existing 40 V3.1 JSONs and saved Walmart data
# NO RETRAINING
# ============================================================

import pandas as pd
import numpy as np
import json
import zipfile
from pathlib import Path
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
SOURCE = BASE / "rag_cases_v3_1"
OUT = BASE / "rag_cases_v3_2"
OUT.mkdir(parents=True, exist_ok=True)

# 1. Read original historical sales and saved ET forecasts
sales = pd.read_csv(BASE / "train.csv", parse_dates=["Date"])
pred = pd.read_csv(
    BASE / "final_test_results.csv",
    parse_dates=["Date"]
)
features = pd.read_csv(
    BASE / "features.csv",
    parse_dates=["Date"]
)

required_sales = {"Store", "Dept", "Date", "Weekly_Sales"}
required_pred = {
    "Store", "Dept", "Date",
    "Actual_Sales", "Expected_Sales"
}

assert required_sales.issubset(sales.columns)
assert required_pred.issubset(pred.columns)
assert not sales.duplicated(["Store", "Dept", "Date"]).any()
assert not pred.duplicated(["Store", "Dept", "Date"]).any()
assert not features.duplicated(["Store", "Date"]).any()

files = sorted(SOURCE.glob("*.json"))
assert len(files) == 40, f"Expected 40 JSONs; found {len(files)}"

# Historical comparison thresholds are exploratory rules,
# not statistically validated anomaly thresholds.
RECENT_LOW_RATIO = 0.80
FORECAST_HIGH_RATIO = 2.00
MIN_PEER_STORES = 5

def num(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except (TypeError, ValueError):
        return None

def pct_change(current, baseline):
    a, b = num(current), num(baseline)
    if a is None or b is None or b <= 0:
        return None
    return round(100 * (a / b - 1), 2)

def median_or_none(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return round(float(s.median()), 2) if len(s) else None

def history_review(store, dept, date, actual, expected):
    # Same store, same department; strictly before case date
    h = sales.loc[
        (sales["Store"] == store) &
        (sales["Dept"] == dept) &
        (sales["Date"] < date)
    ].sort_values("Date").copy()

    # Exact previous calendar weeks, never future data
    h["Days_Before"] = (date - h["Date"]).dt.days

    recent = h.loc[
        (h["Days_Before"] >= 7) &
        (h["Days_Before"] <= 56) &
        (h["Days_Before"] % 7 == 0)
    ]

    prev4 = recent[recent["Days_Before"] <= 28]
    prev8 = recent

    def recent_summary(frame, minimum):
        values = frame["Weekly_Sales"].dropna()
        enough = len(values) >= minimum

        return {
            "observations": int(len(values)),
            "median_actual_sales": (
                median_or_none(values) if enough else None
            ),
            "current_actual_vs_median_pct": (
                pct_change(actual, median_or_none(values))
                if enough else None
            ),
            "sufficient_history": bool(enough)
        }

    r4 = recent_summary(prev4, 3)
    r8 = recent_summary(prev8, 5)

    # Same ISO week +/- 1 in prior calendar years only
    case_week = int(date.isocalendar().week)
    h["Year"] = h["Date"].dt.year
    h["ISO_Week"] = h["Date"].dt.isocalendar().week.astype(int)

    seasonal = h.loc[
        (h["Year"] < date.year) &
        ((h["ISO_Week"] - case_week).abs() <= 1)
    ]

    seasonal_values = seasonal["Weekly_Sales"].dropna()
    seasonal_median = (
        median_or_none(seasonal_values)
        if len(seasonal_values) >= 2 else None
    )

    recent_baseline = r4["median_actual_sales"]

    # Exploratory forecast plausibility checks
    forecast_vs_recent = (
        num(expected) / recent_baseline
        if recent_baseline is not None and
        recent_baseline > 0 else None
    )

    forecast_vs_seasonal = (
        num(expected) / seasonal_median
        if seasonal_median is not None and
        seasonal_median > 0 else None
    )

    supported_checks = [
        x for x in [
            r4["current_actual_vs_median_pct"],
            pct_change(actual, seasonal_median)
        ] if x is not None
    ]

    sales_below_both = (
        len(supported_checks) == 2 and
        all(x <= -20 for x in supported_checks)
    )

    forecast_inflation_checks = [
        x for x in [
            forecast_vs_recent,
            forecast_vs_seasonal
        ] if x is not None
    ]

    possible_overprediction = any(
        x >= FORECAST_HIGH_RATIO
        for x in forecast_inflation_checks
    )

    if possible_overprediction:
        diagnostic = "Possible forecast overprediction"
    elif sales_below_both:
        diagnostic = "Historical comparisons support sales decline"
    elif len(supported_checks) == 2:
        diagnostic = "Mixed or limited evidence of sales decline"
    else:
        diagnostic = "Insufficient historical comparison evidence"

    return {
        "recent_4_weeks": r4,
        "recent_8_weeks": r8,
        "same_season_previous_years": {
            "method": "Same ISO week +/- 1 in years before case",
            "observations": int(len(seasonal_values)),
            "years": sorted(
                [int(y) for y in seasonal["Year"].unique()]
            ),
            "median_actual_sales": seasonal_median,
            "current_actual_vs_seasonal_pct":
                pct_change(actual, seasonal_median)
        },
        "forecast_plausibility": {
            "expected_divided_by_recent_median":
                num(forecast_vs_recent),
            "expected_divided_by_seasonal_median":
                num(forecast_vs_seasonal),
            "possible_overprediction": bool(
                possible_overprediction
            ),
            "historical_diagnostic": diagnostic
        }
    }

def peer_review(store, dept, date):
    peer = pred.loc[
        (pred["Dept"] == dept) &
        (pred["Date"] == date) &
        (pred["Store"] != store) &
        (pred["Expected_Sales"] > 0)
    ].copy()

    peer["Overpredicted"] = (
        peer["Actual_Sales"] < peer["Expected_Sales"]
    )

    n = int(peer["Store"].nunique())

    return {
        "other_stores_compared": n,
        "overprediction_rate_pct": (
            round(100 * peer["Overpredicted"].mean(), 2)
            if n >= MIN_PEER_STORES else None
        ),
        "interpretation": (
            "A high cross-store overprediction rate may "
            "indicate shared forecasting bias or a broader "
            "sales pattern; it does not establish causality."
        ),
        "sufficient_peers": n >= MIN_PEER_STORES
    }

def external_review(store, date):
    current = features.loc[
        (features["Store"] == store) &
        (features["Date"] == date)
    ]

    if current.empty:
        return {"status": "Context unavailable"}

    past = features.loc[
        (features["Store"] == store) &
        (features["Date"] < date) &
        (features["Date"] >= date - pd.Timedelta(weeks=4))
    ]

    result = {}
    cols = [
        "Temperature", "Fuel_Price", "CPI", "Unemployment",
        "MarkDown1", "MarkDown2", "MarkDown3",
        "MarkDown4", "MarkDown5"
    ]

    for col in cols:
        if col not in features.columns:
            continue

        cur = num(current.iloc[0][col])
        old = pd.to_numeric(
            past[col], errors="coerce"
        ).dropna()

        med = median_or_none(old) if len(old) >= 2 else None

        result[col] = {
            "current": cur,
            "prior_4_week_median": med,
            "prior_recorded_observations": int(len(old)),
            "difference": (
                round(cur - med, 3)
                if cur is not None and med is not None
                else None
            )
        }

    return result

# 2. Enrich each JSON
rows = []
written = []

for path in files:
    with open(path, encoding="utf-8") as f:
        case = json.load(f)

    sw = case["store_week"]
    store = int(sw["store_id"])
    date = pd.Timestamp(sw["date"])

    dept_entries = case["sales_evidence"][
        "global_mad_flagged_departments"
    ]["departments"]

    for d in dept_entries:
        dept = int(d["department_id"])

        actual = d["actual_weekly_sales"]
        expected = d["expected_weekly_sales_extra_trees"]

        d["historical_validation"] = history_review(
            store, dept, date, actual, expected
        )

        d["cross_store_validation"] = peer_review(
            store, dept, date
        )

    possible_overprediction = [
        int(d["department_id"])
        for d in dept_entries
        if d["historical_validation"][
            "forecast_plausibility"
        ]["possible_overprediction"]
    ]

    supported_decline = [
        int(d["department_id"])
        for d in dept_entries
        if d["historical_validation"][
            "forecast_plausibility"
        ]["historical_diagnostic"] ==
        "Historical comparisons support sales decline"
    ]

    # Root-cause verification cannot be inferred from forecasts
    case["root_cause_assessment"] = {
        "verification_status": "NOT VERIFIED",
        "verified_business_root_cause": None,
        "evidence_supported_observations": {
            "hybrid_screening_passed": True,
            "departments_with_historical_decline_support":
                supported_decline,
            "departments_with_possible_forecast_overprediction":
                possible_overprediction
        },
        "candidate_hypotheses_not_confirmed": [
            "Forecast seasonality or model overprediction",
            "Inventory shortage or replenishment issue",
            "Markdown timing or promotion execution issue",
            "Broader market or demand shift",
            "Data quality or sales reporting issue"
        ],
        "evidence_required_for_verification": [
            {
                "hypothesis": "Inventory shortage",
                "required_data":
                    "Department or SKU inventory, stockouts, "
                    "replenishment and availability records"
            },
            {
                "hypothesis": "Promotion execution issue",
                "required_data":
                    "Promotion calendar, item-level discounts, "
                    "actual selling prices and execution logs"
            },
            {
                "hypothesis": "Forecast model error",
                "required_data":
                    "Seasonal backtesting, analogous-store "
                    "comparisons and model residual review"
            },
            {
                "hypothesis": "Demand or external shock",
                "required_data":
                    "Local events, market demand evidence "
                    "and comparable unaffected locations"
            },
            {
                "hypothesis": "Data quality issue",
                "required_data":
                    "Source transaction audit and corrections"
            }
        ],
        "agent_instruction": (
            "Use retrieved documents to evaluate hypotheses. "
            "Only report a verified cause when independent "
            "source evidence directly supports it. "
            "Otherwise state that the cause remains unknown."
        )
    }

    case["recent_external_comparison"] = external_review(
        store, date
    )

    case["schema_version"] = "3.2"
    case["validation_notes"] = {
        "recent_reference":
            "Prior 4 and 8 exact calendar weeks, strictly "
            "earlier than the case date.",
        "seasonal_reference":
            "Prior-year same ISO week +/- one week; "
            "not necessarily identical holidays.",
        "peer_reference":
            "Other stores in the same department and date.",
        "heuristics":
            "20% below both historical baselines and 2x "
            "forecast-to-baseline are exploratory diagnostic "
            "rules, not validated detection thresholds.",
        "causality":
            "These comparisons cannot verify a business cause."
    }

    # 3. Save individual enriched case
    outpath = OUT / path.name

    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(
            case, f, indent=2, ensure_ascii=False,
            allow_nan=False
        )

    written.append(outpath)

    rows.append({
        "Case_ID": case["case_id"],
        "Priority_Rank": sw["priority_rank"],
        "Store": store,
        "Date": sw["date"],
        "Flagged_Departments": len(dept_entries),
        "Historical_Decline_Supported_Depts":
            len(supported_decline),
        "Possible_Overprediction_Depts":
            len(possible_overprediction),
        "Root_Cause_Status": "NOT VERIFIED"
    })

# 4. Save summary and ZIP
summary = pd.DataFrame(rows).sort_values("Priority_Rank")

summary.to_csv(
    BASE / "rag_cases_v3_2_validation_summary.csv",
    index=False
)

zip_path = BASE / "walmart_rag_cases_v3_2.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in written:
        z.write(path, arcname=f"rag_cases_v3_2/{path.name}")

with zipfile.ZipFile(zip_path) as z:
    names = [n for n in z.namelist() if n.endswith(".json")]
    assert len(names) == 40
    for name in names:
        json.loads(z.read(name))

print("V3.2 COMPLETE")
print("Cases enriched:", len(written))
print("Cases with possible forecast overprediction:",
      int((summary["Possible_Overprediction_Depts"] > 0).sum()))
print("Cases with historical decline support:",
      int((summary["Historical_Decline_Supported_Depts"] > 0).sum()))
print("Verified root causes: 0 (not independently assessed)")
print("ZIP:", zip_path)

display(summary.head(10))


V3.2 COMPLETE
Cases enriched: 40
Cases with possible forecast overprediction: 30
Cases with historical decline support: 37
Verified root causes: 0 (not independently assessed)
ZIP: /content/drive/MyDrive/Walmart Recruiting Sales Data/walmart_rag_cases_v3_2.zip


,Case_ID,Priority_Rank,Store,Date,Flagged_Departments,Historical_Decline_Supported_Depts,Possible_Overprediction_Depts,Root_Cause_Status
0,Store_39_2012-09-07,1,39,2012-09-07,10,0,6,NOT VERIFIED
1,Store_28_2012-08-31,2,28,2012-08-31,11,4,1,NOT VERIFIED
2,Store_12_2012-09-07,3,12,2012-09-07,11,3,4,NOT VERIFIED
3,Store_6_2012-09-14,4,6,2012-09-14,14,2,2,NOT VERIFIED
4,Store_4_2012-08-24,5,4,2012-08-24,12,4,2,NOT VERIFIED
5,Store_8_2012-09-07,6,8,2012-09-07,14,3,5,NOT VERIFIED
6,Store_10_2012-08-10,7,10,2012-08-10,9,4,1,NOT VERIFIED
7,Store_10_2012-08-31,8,10,2012-08-31,7,3,1,NOT VERIFIED
8,Store_4_2012-08-31,9,4,2012-08-31,7,2,1,NOT VERIFIED
9,Store_4_2012-09-07,10,4,2012-09-07,16,7,2,NOT VERIFIED


**V3.3 — Separate diagnostic signals:** Separated recent sales declines, seasonal declines, potential forecast overprediction, and widespread peer-store overprediction into independent indicators.

In [15]:

# ============================================================
# STAGE 17 — V3.3 RAG EXPORT
# Separate sales decline and forecast reliability signals
# No retraining | Preserves V3.2 | Same 40 hybrid cases
# ============================================================

import json
import zipfile
import pandas as pd
from pathlib import Path
from IPython.display import display

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
SOURCE = BASE / "rag_cases_v3_2"
OUT = BASE / "rag_cases_v3_3"
OUT.mkdir(parents=True, exist_ok=True)

source_files = sorted(SOURCE.glob("*.json"))
assert len(source_files) == 40, (
    f"Expected 40 V3.2 cases, found {len(source_files)}"
)

summary_rows = []
written = []

def below_threshold(value, threshold):
    return (
        value is not None
        and float(value) <= threshold
    )

for path in source_files:

    with open(path, encoding="utf-8") as f:
        case = json.load(f)

    departments = case["sales_evidence"][
        "global_mad_flagged_departments"
    ]["departments"]

    for d in departments:

        hist = d["historical_validation"]
        recent = hist["recent_4_weeks"]
        seasonal = hist["same_season_previous_years"]
        plausibility = hist["forecast_plausibility"]
        peer = d["cross_store_validation"]

        recent_pct = recent.get(
            "current_actual_vs_median_pct"
        )
        seasonal_pct = seasonal.get(
            "current_actual_vs_seasonal_pct"
        )
        peer_rate = peer.get(
            "overprediction_rate_pct"
        )

        recent_decline = (
            recent.get("sufficient_history", False)
            and below_threshold(recent_pct, -20)
        )

        seasonal_decline = below_threshold(
            seasonal_pct, -20
        )

        possible_overprediction = bool(
            plausibility.get(
                "possible_overprediction", False
            )
        )

        widespread_peer_overprediction = (
            peer.get("sufficient_peers", False)
            and peer_rate is not None
            and float(peer_rate) >= 70
        )

        # Multiple signals can be true simultaneously
        d["diagnostic_signals"] = {
            "recent_sales_decline_20pct": bool(
                recent_decline
            ),
            "prior_year_seasonal_decline_20pct": bool(
                seasonal_decline
            ),
            "possible_forecast_overprediction": bool(
                possible_overprediction
            ),
            "widespread_cross_store_overprediction_70pct":
                bool(widespread_peer_overprediction),
            "interpretation": (
                "These are independent exploratory "
                "signals. Multiple signals may occur "
                "simultaneously. None establishes "
                "a verified business root cause."
            )
        }

    signals = [
        d["diagnostic_signals"]
        for d in departments
    ]

    recent_n = sum(
        s["recent_sales_decline_20pct"]
        for s in signals
    )

    seasonal_n = sum(
        s["prior_year_seasonal_decline_20pct"]
        for s in signals
    )

    overprediction_n = sum(
        s["possible_forecast_overprediction"]
        for s in signals
    )

    widespread_n = sum(
        s["widespread_cross_store_overprediction_70pct"]
        for s in signals
    )

    case["case_diagnostics"] = {
        "departments_with_recent_sales_decline":
            recent_n,
        "departments_with_prior_year_seasonal_decline":
            seasonal_n,
        "departments_with_possible_forecast_overprediction":
            overprediction_n,
        "departments_with_widespread_peer_overprediction":
            widespread_n,
        "overall_interpretation": (
            "Recent sales decline, seasonal behavior, "
            "and forecast reliability are evaluated "
            "separately to avoid confusing a business "
            "anomaly with potential forecasting bias."
        )
    }

    # Preserve unverified status
    case["root_cause_assessment"][
        "verification_status"
    ] = "NOT VERIFIED"

    case["root_cause_assessment"][
        "verified_business_root_cause"
    ] = None

    case["schema_version"] = "3.3"

    outpath = OUT / path.name

    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(
            case,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False
        )

    written.append(outpath)

    summary_rows.append({
        "Case_ID": case["case_id"],
        "Priority_Rank":
            case["store_week"]["priority_rank"],
        "Store": case["store_week"]["store_id"],
        "Date": case["store_week"]["date"],
        "Type": case["store_week"]["store_type"],
        "Flagged_Departments": len(departments),
        "Recent_Decline_Depts": recent_n,
        "Seasonal_Decline_Depts": seasonal_n,
        "Possible_Overprediction_Depts":
            overprediction_n,
        "Widespread_Peer_Overprediction_Depts":
            widespread_n,
        "Root_Cause_Status": "NOT VERIFIED"
    })

# ============================================================
# SAVE SUMMARY
# ============================================================

summary = pd.DataFrame(summary_rows).sort_values(
    "Priority_Rank"
)

summary_path = (
    BASE / "rag_cases_v3_3_diagnostic_summary.csv"
)

summary.to_csv(summary_path, index=False)

# ============================================================
# ZIP 40 JSON FILES
# ============================================================

zip_path = BASE / "walmart_rag_cases_v3_3.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zf:

    for path in written:
        zf.write(
            path,
            arcname=f"rag_cases_v3_3/{path.name}"
        )

# ============================================================
# FINAL VALIDATION
# ============================================================

with zipfile.ZipFile(zip_path) as zf:

    json_names = [
        name for name in zf.namelist()
        if name.endswith(".json")
    ]

    assert len(json_names) == 40

    for name in json_names:
        json.loads(zf.read(name))

assert summary["Case_ID"].is_unique
assert set(summary["Priority_Rank"]) == set(range(1, 41))

print("=" * 45)
print("V3.3 RAG EXPORT COMPLETE")
print("=" * 45)
print("JSON cases:", len(written))
print("Stores represented:", summary["Store"].nunique())
print(
    "Cases with recent sales decline:",
    int((summary["Recent_Decline_Depts"] > 0).sum())
)
print(
    "Cases with potential forecast overprediction:",
    int((summary["Possible_Overprediction_Depts"] > 0).sum())
)
print(
    "Cases with widespread peer overprediction:",
    int(
        (summary[
            "Widespread_Peer_Overprediction_Depts"
        ] > 0).sum()
    )
)
print("Verified business root causes: 0")
print("JSON folder:", OUT)
print("Summary:", summary_path)
print("ZIP:", zip_path)

display(summary.head(10))


V3.3 RAG EXPORT COMPLETE
JSON cases: 40
Stores represented: 18
Cases with recent sales decline: 40
Cases with potential forecast overprediction: 30
Cases with widespread peer overprediction: 39
Verified business root causes: 0
JSON folder: /content/drive/MyDrive/Walmart Recruiting Sales Data/rag_cases_v3_3
Summary: /content/drive/MyDrive/Walmart Recruiting Sales Data/rag_cases_v3_3_diagnostic_summary.csv
ZIP: /content/drive/MyDrive/Walmart Recruiting Sales Data/walmart_rag_cases_v3_3.zip


,Case_ID,Priority_Rank,Store,Date,Type,Flagged_Departments,Recent_Decline_Depts,Seasonal_Decline_Depts,Possible_Overprediction_Depts,Widespread_Peer_Overprediction_Depts,Root_Cause_Status
0,Store_39_2012-09-07,1,39,2012-09-07,A,10,9,1,6,8,NOT VERIFIED
1,Store_28_2012-08-31,2,28,2012-08-31,A,11,6,8,1,7,NOT VERIFIED
2,Store_12_2012-09-07,3,12,2012-09-07,B,11,11,3,4,7,NOT VERIFIED
3,Store_6_2012-09-14,4,6,2012-09-14,A,14,11,4,2,8,NOT VERIFIED
4,Store_4_2012-08-24,5,4,2012-08-24,A,12,10,4,2,3,NOT VERIFIED
5,Store_8_2012-09-07,6,8,2012-09-07,A,14,9,5,5,10,NOT VERIFIED
6,Store_10_2012-08-10,7,10,2012-08-10,B,9,7,5,1,2,NOT VERIFIED
7,Store_10_2012-08-31,8,10,2012-08-31,B,7,5,4,1,3,NOT VERIFIED
8,Store_4_2012-08-31,9,4,2012-08-31,A,7,7,3,1,2,NOT VERIFIED
9,Store_4_2012-09-07,10,4,2012-09-07,A,16,11,10,2,8,NOT VERIFIED


**Interactive store and department dashboard**

Every Store

In [16]:

# ============================================================
# STAGE 18 — INTERACTIVE WALMART ANOMALY DASHBOARD
# All 45 stores | All 675 store-weeks | No retraining
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
REP = BASE / "michaela_replication"

# 1. Load saved model results
store_data = pd.read_csv(
    BASE / "et_mad_store_week_with_factors.csv",
    parse_dates=["Date"]
)

dept_data = pd.read_csv(
    BASE / "et_mad_department_week.csv",
    parse_dates=["Date"]
)

if_data = pd.read_csv(
    REP / "michaela_store_week_comparison.csv",
    parse_dates=["Date"]
)

hybrid_data = pd.read_csv(
    BASE / "FINAL_hybrid_all_cases.csv",
    parse_dates=["Date"]
)

def bool_col(series):
    return series.astype(str).str.lower().isin(
        ["true", "1"]
    )

store_data["ET_MAD_Flag"] = bool_col(
    store_data["ET_MAD_Flag"]
)

dept_data["Flag_GlobalMAD"] = bool_col(
    dept_data["Flag_GlobalMAD"]
)

if_data["IF_Flag"] = bool_col(
    if_data["IF_Flag"]
)

# Merge IF scores without overwriting ET fields
store_data = store_data.merge(
    if_data[[
        "Store", "Date", "IF_Flag", "IF_Anomaly_Score"
    ]],
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

hybrid_data["Hybrid_Flag"] = bool_col(
    hybrid_data["Hybrid_Flag"]
)

# Identify which 40 store-weeks are selected
hybrid_keys = hybrid_data[[
    "Store", "Date", "Priority_Rank"
]].copy()

store_data = store_data.merge(
    hybrid_keys,
    on=["Store", "Date"],
    how="left",
    validate="one_to_one"
)

# Identify observed sales and forecast fields
def find_column(df, names):
    lookup = {
        str(c).lower(): c for c in df.columns
    }
    for name in names:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

ACTUAL = find_column(
    dept_data, ["Actual_Sales", "Weekly_Sales"]
)
EXPECTED = find_column(
    dept_data, ["Expected_Sales", "Predicted_Sales"]
)

assert ACTUAL is not None
assert EXPECTED is not None

# 2. Store / date selectors
store_widget = widgets.Dropdown(
    options=sorted(store_data["Store"].unique()),
    value=39,
    description="Store:",
    style={"description_width": "initial"}
)

date_widget = widgets.Dropdown(
    description="Week:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

out = widgets.Output()

def update_dates(*args):
    store = store_widget.value
    dates = (
        store_data.loc[
            store_data["Store"] == store, "Date"
        ].sort_values().unique()
    )
    options = [
        (
            pd.Timestamp(d).strftime("%Y-%m-%d"),
            pd.Timestamp(d)
        )
        for d in dates
    ]
    date_widget.options = options

    preferred = pd.Timestamp("2012-09-07")
    if store == 39 and preferred in [v for _, v in options]:
        date_widget.value = preferred

def draw_dashboard(*args):
    if date_widget.value is None:
        return

    with out:
        clear_output(wait=True)

        store = store_widget.value
        date = pd.Timestamp(date_widget.value)

        sw = store_data.loc[
            (store_data["Store"] == store) &
            (store_data["Date"] == date)
        ]

        if sw.empty:
            print("No store-week results available.")
            return

        sw = sw.iloc[0]

        dw = dept_data.loc[
            (dept_data["Store"] == store) &
            (dept_data["Date"] == date)
        ].copy()

        dw["Modeled_Shortfall"] = pd.to_numeric(
            dw["Dollar_Shortfall"], errors="coerce"
        )

        flagged_dw = dw.loc[
            dw["Flag_GlobalMAD"]
        ].sort_values(
            "Modeled_Shortfall", ascending=False
        )

        print("=" * 60)
        print(
            f"STORE {store} | {date:%B %d, %Y} "
            f"| TYPE {sw['Type']}"
        )
        print("=" * 60)

        print(
            f"IF flagged: {sw['IF_Flag']} "
            f"| IF score: {sw['IF_Anomaly_Score']:.5f}"
        )
        print(
            f"ET Global MAD flagged: {sw['ET_MAD_Flag']}"
        )

        rank = sw.get("Priority_Rank")
        print(
            "Hybrid priority:",
            int(rank) if pd.notna(rank)
            else "Not in final 40"
        )

        print(
            f"ET net modeled shortfall: "
            f"${sw['ET_Net_Shortfall']:,.2f}"
        )
        print(
            f"Flagged department shortfall: "
            f"${sw['ET_Flagged_Shortfall']:,.2f}"
        )

        print(
            f"Flagged departments: {len(flagged_dw)} "
            f"/ {len(dw)} eligible"
        )

        # 3. Chart: actual vs predicted for flagged depts
        if len(flagged_dw):
            top = flagged_dw.head(15).sort_values(
                "Modeled_Shortfall", ascending=True
            )

            fig, ax = plt.subplots(
                figsize=(11, max(4, len(top) * 0.45))
            )

            positions = np.arange(len(top))
            bar_height = 0.38

            ax.barh(
                positions - bar_height / 2,
                top[ACTUAL],
                height=bar_height,
                label="Actual sales"
            )

            ax.barh(
                positions + bar_height / 2,
                top[EXPECTED],
                height=bar_height,
                label="ET expected sales"
            )

            ax.set_yticks(positions)
            ax.set_yticklabels(
                ["Dept " + str(int(x))
                 for x in top["Dept"]]
            )
            ax.set_xlabel("Weekly sales ($)")
            ax.set_title(
                "Flagged departments: actual vs expected"
            )
            ax.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("No Global MAD-flagged departments.")

        # 4. Table: ALL eligible departments
        print("\nALL ELIGIBLE DEPARTMENTS")

        display_cols = [
            "Dept", ACTUAL, EXPECTED,
            "Dollar_Shortfall", "Shortfall_Pct",
            "Flag_GlobalMAD"
        ]

        available = [
            c for c in display_cols
            if c in dw.columns
        ]

        display(
            dw[available].sort_values(
                ["Flag_GlobalMAD", "Dollar_Shortfall"],
                ascending=[False, False]
            ).reset_index(drop=True)
        )

        # 5. Context information
        print("\nMARKDOWN AND EXTERNAL CONDITIONS")

        context_cols = [
            "MarkDown1", "MarkDown2", "MarkDown3",
            "MarkDown4", "MarkDown5",
            "Recorded_Markdown_Total",
            "Temperature", "Fuel_Price",
            "CPI", "Unemployment"
        ]

        display(
            pd.DataFrame([{
                c: sw.get(c, np.nan)
                for c in context_cols
            }]).T.rename(columns={0: "Value"})
        )

        # 6. All store-week source values
        print("\nCOMPLETE STORE-WEEK RECORD")
        display(
            sw.to_frame(name="Value")
        )

store_widget.observe(update_dates, names="value")
store_widget.observe(draw_dashboard, names="value")
date_widget.observe(draw_dashboard, names="value")

update_dates()

display(
    widgets.HTML(
        "<h3>Walmart Anomaly Investigation Dashboard</h3>"
    ),
    widgets.HBox([store_widget, date_widget]),
    out
)

draw_dashboard()


HTML(value='<h3>Walmart Anomaly Investigation Dashboard</h3>')

Output()

V3.3 dashboard

In [17]:

# ============================================================
# STAGE 19 — V3.3 INTERACTIVE CASE EXPLORER
# Choose among the 40 FINAL hybrid investigation cases
# No retraining | Reads existing V3.3 JSON files
# ============================================================

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
CASE_FOLDER = BASE / "rag_cases_v3_3"

json_files = list(CASE_FOLDER.glob("*.json"))
assert len(json_files) == 40, (
    f"Expected 40 V3.3 JSON cases, found {len(json_files)}"
)

# Load all cases
case_data = {}

for path in json_files:
    with open(path, encoding="utf-8") as f:
        case = json.load(f)

    assert case["schema_version"] == "3.3"
    rank = int(case["store_week"]["priority_rank"])
    assert rank not in case_data
    case_data[rank] = case

assert sorted(case_data) == list(range(1, 41))

def money(x):
    return f"${x:,.2f}" if x is not None else "N/A"

def pct(x):
    return f"{x:+.2f}%" if x is not None else "N/A"

# Dropdown labels include rank, store and date
options = []

for rank, case in sorted(case_data.items()):
    sw = case["store_week"]
    options.append((
        f"#{rank:02d} | Store {sw['store_id']} | "
        f"{sw['date']} | Type {sw['store_type']}",
        rank
    ))

case_dropdown = widgets.Dropdown(
    options=options,
    value=1,
    description="V3.3 Case:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px")
)

output = widgets.Output()

def display_case(change=None):

    with output:
        clear_output(wait=True)

        case = case_data[case_dropdown.value]
        sw = case["store_week"]
        det = case["hybrid_detection"]
        sales = case["sales_evidence"]

        agg = sales["eligible_department_aggregate"]
        flagged = sales["global_mad_flagged_departments"]
        departments = flagged["departments"]

        print("=" * 65)
        print(
            f"V3.3 | PRIORITY #{sw['priority_rank']} "
            f"| STORE {sw['store_id']} | {sw['date']}"
        )
        print("=" * 65)

        print(f"Store type: {sw['store_type']}")
        print(f"Isolation Forest: {det['isolation_forest']['flag']}")
        print(
            "IF anomaly score:",
            round(det["isolation_forest"]["anomaly_score"], 5)
        )
        print(f"Extra Trees MAD: {det['extra_trees_global_mad']['flag']}")

        print("\nSALES SUMMARY")
        print("ET expected:", money(agg["expected_sales"]))
        print("Actual sales:", money(agg["actual_sales"]))
        print("Net modeled gap:", money(agg["net_modeled_shortfall"]))
        print(
            "Flagged department gap:",
            money(flagged["sum_positive_modeled_shortfalls"])
        )
        print("Flagged department count:", len(departments))

        # -----------------------------------------------
        # ACTUAL VS EXPECTED CHART
        # -----------------------------------------------

        df = pd.DataFrame([{
            "Dept": int(d["department_id"]),
            "Actual": d["actual_weekly_sales"],
            "Expected": d["expected_weekly_sales_extra_trees"],
            "Shortfall": d["modeled_dollar_shortfall"]
        } for d in departments])

        df = df.sort_values("Shortfall", ascending=False)

        plot_df = df.head(15).iloc[::-1]

        fig, ax = plt.subplots(
            figsize=(11, max(4, len(plot_df) * 0.5))
        )

        y = np.arange(len(plot_df))

        ax.barh(
            y - 0.2,
            plot_df["Actual"],
            height=0.38,
            label="Actual sales"
        )

        ax.barh(
            y + 0.2,
            plot_df["Expected"],
            height=0.38,
            label="Extra Trees expected"
        )

        ax.set_yticks(y)
        ax.set_yticklabels(
            [f"Dept {x}" for x in plot_df["Dept"]]
        )

        ax.set_xlabel("Weekly sales ($)")
        ax.set_title(
            "Global MAD flagged departments: Actual vs Expected"
        )
        ax.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------
        # DEPARTMENT DETAIL TABLE
        # -----------------------------------------------

        print("\nFLAGGED DEPARTMENT RESULTS")

        display(
            df.rename(columns={
                "Dept": "Department",
                "Actual": "Actual Sales",
                "Expected": "Expected Sales",
                "Shortfall": "Modeled Shortfall"
            }).reset_index(drop=True).round(2)
        )

        # -----------------------------------------------
        # V3.3 HISTORICAL / SEASONAL DIAGNOSTICS
        # -----------------------------------------------

        diagnostic_rows = []

        for d in departments:
            h = d["historical_validation"]
            s = d["diagnostic_signals"]
            peer = d["cross_store_validation"]

            diagnostic_rows.append({
                "Dept": int(d["department_id"]),
                "Vs Prior 4W %":
                    h["recent_4_weeks"]["current_actual_vs_median_pct"],
                "Vs Prior 8W %":
                    h["recent_8_weeks"]["current_actual_vs_median_pct"],
                "Vs Prior Years %":
                    h["same_season_previous_years"][
                        "current_actual_vs_seasonal_pct"
                    ],
                "Recent Decline":
                    s["recent_sales_decline_20pct"],
                "Forecast Overprediction Risk":
                    s["possible_forecast_overprediction"],
                "Peer Overprediction %":
                    peer["overprediction_rate_pct"],
                "Widespread Peer Bias":
                    s["widespread_cross_store_overprediction_70pct"]
            })

        print("\nV3.3 HISTORICAL AND FORECAST DIAGNOSTICS")
        display(
            pd.DataFrame(diagnostic_rows)
            .sort_values("Dept")
            .reset_index(drop=True)
            .round(2)
        )

        # -----------------------------------------------
        # V3.3 CASE-LEVEL SIGNAL SUMMARY
        # -----------------------------------------------

        diag = case["case_diagnostics"]

        print("\nCASE DIAGNOSTIC SUMMARY")

        display(pd.DataFrame([{
            "Recent Decline Depts":
                diag["departments_with_recent_sales_decline"],
            "Seasonal Decline Depts":
                diag["departments_with_prior_year_seasonal_decline"],
            "Possible Forecast Overprediction Depts":
                diag["departments_with_possible_forecast_overprediction"],
            "Widespread Peer Overprediction Depts":
                diag["departments_with_widespread_peer_overprediction"]
        }]))

        # -----------------------------------------------
        # MARKDOWN AND ECONOMIC FACTORS
        # -----------------------------------------------

        markdown = case["markdown_context"]
        external = case["external_context"]

        print("\nMARKDOWN CONTEXT")

        md_rows = [
            {"Field": k, "Recorded Value": v}
            for k, v in markdown["markdown_fields"].items()
        ]

        md_rows.append({
            "Field": "Recorded Markdown Total",
            "Recorded Value": markdown["recorded_markdown_total"]
        })

        display(pd.DataFrame(md_rows).round(2))

        print("\nEXTERNAL CONDITIONS")

        display(pd.DataFrame([
            {"Factor": "Temperature",
             "Value": external["temperature"]},
            {"Factor": "Fuel Price",
             "Value": external["fuel_price"]},
            {"Factor": "CPI",
             "Value": external["cpi"]},
            {"Factor": "Unemployment",
             "Value": external["unemployment"]}
        ]).round(3))

        print("\nROOT CAUSE STATUS")
        print(
            case["root_cause_assessment"]["verification_status"]
        )
        print(
            "These are model-based investigation signals, "
            "not confirmed markdown-related causes."
        )

case_dropdown.observe(display_case, names="value")

display(
    widgets.HTML("<h3>Walmart V3.3 Investigation Dashboard</h3>"),
    case_dropdown,
    output
)

display_case()


HTML(value='<h3>Walmart V3.3 Investigation Dashboard</h3>')

Dropdown(description='V3.3 Case:', layout=Layout(width='550px'), options=(('#01 | Store 39 | 2012-09-07 | Type…

Output()

**Enhanced Dashboard**

In [18]:

# ============================================================
# STAGE 20 — ENHANCED V3.3 CASE EXPLORER
# Historical trends + Warning signals + Export button
# NO RETRAINING
# ============================================================

import json
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

BASE = Path("/content/drive/MyDrive/Walmart Recruiting Sales Data")
CASE_DIR = BASE / "rag_cases_v3_3"
EXPORT_DIR = BASE / "v3_3_dashboard_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD V3.3 CASES AND HISTORICAL SALES
# ------------------------------------------------------------

case_files = sorted(CASE_DIR.glob("*.json"))
assert len(case_files) == 40, "Expected 40 V3.3 JSON cases."

cases = {}

for path in case_files:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    assert data["schema_version"] == "3.3"
    rank = int(data["store_week"]["priority_rank"])
    cases[rank] = data

assert sorted(cases) == list(range(1, 41))

sales_path = BASE / "train.csv"

if sales_path.exists():
    history = pd.read_csv(
        sales_path,
        parse_dates=["Date"]
    )
    history = history[
        ["Store", "Dept", "Date", "Weekly_Sales"]
    ].copy()

    history = history.sort_values(
        ["Store", "Dept", "Date"]
    )

    print("Historical sales loaded successfully.")
else:
    history = pd.DataFrame(
        columns=["Store", "Dept", "Date", "Weekly_Sales"]
    )
    print("Warning: train.csv not found; trend will be limited.")

# ------------------------------------------------------------
# 2. FORMAT HELPERS
# ------------------------------------------------------------

def dollar(x):
    if x is None or pd.isna(x):
        return "N/A"
    return f"${float(x):,.2f}"

def percentage(x):
    if x is None or pd.isna(x):
        return "N/A"
    return f"{float(x):+.1f}%"

def warning_label(value):
    return "YES — Review" if value else "No"

# ------------------------------------------------------------
# 3. CREATE DROPDOWNS
# ------------------------------------------------------------

case_options = []

for rank, case in sorted(cases.items()):
    sw = case["store_week"]
    case_options.append((
        f"#{rank:02d} | Store {sw['store_id']} | "
        f"{sw['date']} | Type {sw['store_type']}",
        rank
    ))

case_select = widgets.Dropdown(
    options=case_options,
    value=1,
    description="Case:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="510px")
)

department_select = widgets.Dropdown(
    description="Department:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="240px")
)

export_button = widgets.Button(
    description="Export Selected Case",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="220px")
)

dashboard_output = widgets.Output()
export_output = widgets.Output()

# ------------------------------------------------------------
# 4. UPDATE DEPARTMENTS WHEN CASE CHANGES
# ------------------------------------------------------------

def selected_departments(case):
    return case["sales_evidence"][
        "global_mad_flagged_departments"
    ]["departments"]

def update_departments(change=None):
    case = cases[case_select.value]
    depts = selected_departments(case)

    options = [
        (
            f"Dept {d['department_id']} "
            f"({dollar(d['modeled_dollar_shortfall'])} gap)",
            int(d["department_id"])
        )
        for d in depts
    ]

    department_select.options = options

    if options:
        department_select.value = options[0][1]

# ------------------------------------------------------------
# 5. BUILD HISTORICAL CHART
# ------------------------------------------------------------

def make_trend_chart(case, dept):
    sw = case["store_week"]

    store = int(sw["store_id"])
    date = pd.Timestamp(sw["date"])
    dept_id = int(dept["department_id"])

    # Exactly the preceding 12 calendar weeks
    start = date - pd.Timedelta(weeks=12)

    past = history.loc[
        (history["Store"] == store) &
        (history["Dept"] == dept_id) &
        (history["Date"] >= start) &
        (history["Date"] < date)
    ].sort_values("Date")

    current_actual = dept["actual_weekly_sales"]
    current_expected = dept[
        "expected_weekly_sales_extra_trees"
    ]

    seasonal = dept["historical_validation"][
        "same_season_previous_years"
    ].get("median_actual_sales")

    fig, ax = plt.subplots(figsize=(11, 5))

    # Historical observed sales
    if len(past):
        ax.plot(
            past["Date"],
            past["Weekly_Sales"],
            marker="o",
            linewidth=2,
            label="Historical actual sales"
        )

    # Selected week's observed actual sales
    if current_actual is not None:
        ax.scatter(
            [date],
            [current_actual],
            s=120,
            marker="o",
            label="Selected week: actual",
            zorder=5
        )

    # Selected week's model prediction
    if current_expected is not None:
        ax.scatter(
            [date],
            [current_expected],
            s=150,
            marker="X",
            label="Selected week: ET expected",
            zorder=5
        )

    # Historical same-season reference
    if seasonal is not None:
        ax.axhline(
            seasonal,
            linestyle="--",
            linewidth=1.5,
            label="Prior-year same-season median"
        )

    ax.axvline(
        date,
        linestyle=":",
        alpha=0.6
    )

    ax.set_title(
        f"Store {store} | Dept {dept_id}\n"
        "Recent Actual Sales vs Extra Trees Forecast"
    )
    ax.set_xlabel("Week")
    ax.set_ylabel("Weekly sales ($)")
    ax.grid(alpha=0.2)
    ax.legend(loc="best")

    fig.autofmt_xdate()
    fig.tight_layout()

    return fig

# ------------------------------------------------------------
# 6. BUILD FORECAST COMPARISON CHART
# ------------------------------------------------------------

def make_comparison_chart(case):
    depts = selected_departments(case)

    df = pd.DataFrame([{
        "Dept": int(d["department_id"]),
        "Actual": d["actual_weekly_sales"],
        "Expected": d[
            "expected_weekly_sales_extra_trees"
        ],
        "Shortfall": d["modeled_dollar_shortfall"]
    } for d in depts])

    df = df.sort_values(
        "Shortfall", ascending=False
    ).head(15).iloc[::-1]

    fig, ax = plt.subplots(
        figsize=(10, max(4, len(df) * 0.55))
    )

    y = np.arange(len(df))

    ax.barh(
        y - 0.2,
        df["Actual"],
        height=0.38,
        label="Actual sales"
    )

    ax.barh(
        y + 0.2,
        df["Expected"],
        height=0.38,
        label="ET expected sales"
    )

    ax.set_yticks(y)
    ax.set_yticklabels([
        f"Dept {int(x)}" for x in df["Dept"]
    ])

    ax.set_xlabel("Weekly sales ($)")
    ax.set_title("Global MAD-Flagged Departments")
    ax.legend()
    ax.grid(axis="x", alpha=0.2)

    fig.tight_layout()

    return fig

# ------------------------------------------------------------
# 7. BUILD DEPARTMENT DIAGNOSTIC TABLE
# ------------------------------------------------------------

def make_diagnostic_table(case):
    rows = []

    for d in selected_departments(case):
        h = d["historical_validation"]
        signals = d["diagnostic_signals"]
        peers = d["cross_store_validation"]

        rows.append({
            "Dept": d["department_id"],
            "Actual Sales": d["actual_weekly_sales"],
            "Expected Sales": d[
                "expected_weekly_sales_extra_trees"
            ],
            "Modeled Gap": d["modeled_dollar_shortfall"],
            "Vs Recent 4W %": h["recent_4_weeks"][
                "current_actual_vs_median_pct"
            ],
            "Vs Prior-Year Season %": h[
                "same_season_previous_years"
            ]["current_actual_vs_seasonal_pct"],
            "Recent Decline": signals[
                "recent_sales_decline_20pct"
            ],
            "Forecast Overprediction Risk": signals[
                "possible_forecast_overprediction"
            ],
            "Peer Overprediction %": peers[
                "overprediction_rate_pct"
            ],
            "Widespread Peer Overprediction": signals[
                "widespread_cross_store_overprediction_70pct"
            ]
        })

    return pd.DataFrame(rows).sort_values(
        "Modeled Gap", ascending=False
    )

# ------------------------------------------------------------
# 8. DRAW DASHBOARD
# ------------------------------------------------------------

def draw_dashboard(change=None):
    if department_select.value is None:
        return

    with dashboard_output:
        clear_output(wait=True)

        case = cases[case_select.value]
        sw = case["store_week"]

        depts = selected_departments(case)

        selected_dept = next(
            d for d in depts
            if int(d["department_id"]) ==
            department_select.value
        )

        h = selected_dept["historical_validation"]
        signals = selected_dept["diagnostic_signals"]
        peer = selected_dept["cross_store_validation"]

        print("=" * 65)
        print(
            f"V3.3 | PRIORITY #{sw['priority_rank']} "
            f"| STORE {sw['store_id']} | {sw['date']}"
        )
        print("=" * 65)

        # Display case sales summary
        agg = case["sales_evidence"][
            "eligible_department_aggregate"
        ]

        flagged = case["sales_evidence"][
            "global_mad_flagged_departments"
        ]

        print(
            "Aggregate net modeled gap:",
            dollar(agg["net_modeled_shortfall"])
        )
        print(
            "Flagged department modeled gap:",
            dollar(flagged["sum_positive_modeled_shortfalls"])
        )

        print("\nSELECTED DEPARTMENT")
        print(
            f"Department {selected_dept['department_id']}"
        )
        print(
            "Actual:",
            dollar(selected_dept["actual_weekly_sales"])
        )
        print(
            "ET expected:",
            dollar(selected_dept[
                "expected_weekly_sales_extra_trees"
            ])
        )

        # Warning signals displayed separately
        print("\n" + "=" * 45)
        print("INDEPENDENT DIAGNOSTIC SIGNALS")
        print("=" * 45)

        warning_table = pd.DataFrame([
            {
                "Signal": "Recent sales decline >=20%",
                "Status": warning_label(
                    signals["recent_sales_decline_20pct"]
                ),
                "Evidence": percentage(
                    h["recent_4_weeks"][
                        "current_actual_vs_median_pct"
                    ]
                )
            },
            {
                "Signal": "Possible forecast overprediction",
                "Status": warning_label(
                    signals[
                        "possible_forecast_overprediction"
                    ]
                ),
                "Evidence": (
                    "Exploratory forecast/baseline check"
                )
            },
            {
                "Signal": "Widespread peer overprediction",
                "Status": warning_label(
                    signals[
                        "widespread_cross_store_overprediction_70pct"
                    ]
                ),
                "Evidence": percentage(
                    peer["overprediction_rate_pct"]
                )
            }
        ])

        display(warning_table)

        print(
            "\nThese signals may occur simultaneously. "
            "They do not establish a business root cause."
        )

        # Historical chart
        fig1 = make_trend_chart(case, selected_dept)
        display(fig1)
        plt.close(fig1)

        # All flagged department chart
        fig2 = make_comparison_chart(case)
        display(fig2)
        plt.close(fig2)

        print("\nALL FLAGGED DEPARTMENT DIAGNOSTICS")
        display(make_diagnostic_table(case).round(2))

        print("\nROOT-CAUSE VERIFICATION")
        print(
            case["root_cause_assessment"][
                "verification_status"
            ]
        )

# ------------------------------------------------------------
# 9. EXPORT SELECTED CASE
# ------------------------------------------------------------

def export_selected_case(button):
    with export_output:
        clear_output(wait=True)

        case = cases[case_select.value]
        sw = case["store_week"]

        selected_dept = next(
            d for d in selected_departments(case)
            if int(d["department_id"]) ==
            department_select.value
        )

        case_name = (
            f"rank_{int(sw['priority_rank']):02d}_"
            f"store_{sw['store_id']}_"
            f"{sw['date']}"
        )

        folder = EXPORT_DIR / (
            f"{case_name}_dept_{department_select.value}"
        )
        folder.mkdir(parents=True, exist_ok=True)

        # Chart 1 — Historical trend
        fig1 = make_trend_chart(
            case, selected_dept
        )

        fig1.savefig(
            folder / "historical_sales_trend.png",
            dpi=220, bbox_inches="tight"
        )
        fig1.savefig(
            folder / "historical_sales_trend.pdf",
            bbox_inches="tight"
        )
        plt.close(fig1)

        # Chart 2 — Department comparison
        fig2 = make_comparison_chart(case)

        fig2.savefig(
            folder / "flagged_department_comparison.png",
            dpi=220, bbox_inches="tight"
        )
        fig2.savefig(
            folder / "flagged_department_comparison.pdf",
            bbox_inches="tight"
        )
        plt.close(fig2)

        # Complete numerical table
        make_diagnostic_table(case).to_csv(
            folder / "department_diagnostics.csv",
            index=False
        )

        # Case summary
        agg = case["sales_evidence"][
            "eligible_department_aggregate"
        ]

        flagged_data = case["sales_evidence"][
            "global_mad_flagged_departments"
        ]

        summary = pd.DataFrame([{
            "Case_ID": case["case_id"],
            "Priority_Rank": sw["priority_rank"],
            "Store": sw["store_id"],
            "Date": sw["date"],
            "Type": sw["store_type"],
            "Selected_Dept": department_select.value,
            "IF_Score": case["hybrid_detection"][
                "isolation_forest"
            ]["anomaly_score"],
            "ET_Net_Modeled_Shortfall":
                agg["net_modeled_shortfall"],
            "ET_Flagged_Dept_Shortfall":
                flagged_data[
                    "sum_positive_modeled_shortfalls"
                ],
            "Root_Cause_Status": case[
                "root_cause_assessment"
            ]["verification_status"]
        }])

        summary.to_csv(
            folder / "case_summary.csv",
            index=False
        )

        # Original enriched JSON for RAG traceability
        with open(
            folder / "case_evidence_v3_3.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                case, f, indent=2,
                ensure_ascii=False, allow_nan=False
            )

        # ZIP all exported files for this case
        zip_path = EXPORT_DIR / f"{folder.name}.zip"

        with zipfile.ZipFile(
            zip_path, "w", zipfile.ZIP_DEFLATED
        ) as zf:
            for file in folder.iterdir():
                if file.is_file():
                    zf.write(file, arcname=file.name)

        print("Export completed!")
        print("Case:", case["case_id"])
        print(
            "Selected department:",
            department_select.value
        )
        print("Saved folder:", folder)
        print("ZIP:", zip_path)

# ------------------------------------------------------------
# 10. CONNECT INTERACTIVE CONTROLS
# ------------------------------------------------------------

case_select.observe(
    update_departments, names="value"
)

department_select.observe(
    draw_dashboard, names="value"
)

export_button.on_click(export_selected_case)

update_departments()

display(
    widgets.HTML(
        "<h3>Walmart V3.3 — Enhanced Case Explorer</h3>"
    ),
    widgets.HBox([
        case_select,
        department_select
    ]),
    export_button,
    dashboard_output,
    export_output
)

draw_dashboard()


Historical sales loaded successfully.


HTML(value='<h3>Walmart V3.3 — Enhanced Case Explorer</h3>')

Button(button_style='success', description='Export Selected Case', icon='download', layout=Layout(width='220px…

Output()

Output()

In Store 39, Department 3, our Extra Trees model predicted approximately $76,609 in weekly sales, while actual sales were $27,998, triggering a Global MAD anomaly. Historical analysis revealed a substantial increase in sales during July and August followed by a sharp September decline. Since current sales remained above the same seasonal period in prior years, and Extra Trees overpredicted Department 3 in many other stores, the results suggest a possible seasonal forecasting issue rather than confirmed markdown-related underperformance. This demonstrates why our hybrid system includes historical validation before passing evidence to the RAG agent.